# SafeSort-RL

### Language-conditioned hazardous-waste sorting with Genesis, Franka Panda, PPO, and AMD ROCm

This notebook is the final working record of the SafeSort-RL project. It brings the environment, model loading, evaluation, hardware checks, and the final four-object demonstration into one place so the project can be reviewed without jumping between separate scripts.

The notebook is intentionally self-contained. The only project file it expects at runtime is a trained PPO checkpoint inside the `models/` folder.

---

## What is included

| Part | Purpose |
|---|---|
| AMD ROCm checks | Confirms that PyTorch and Genesis can see the AMD GPU |
| Genesis environment | Builds the Franka Panda workcell, objects, and destination bins |
| PPO checkpoint loading | Loads the strongest available SafeSort model |
| Balanced evaluation | Measures performance across all four instructions |
| Random baseline | Shows how the task performs without a learned policy |
| Instruction-swap probe | Checks whether changing the command changes the selected target |
| Final showcase | Records a clear four-object pick-and-place demonstration |
| Results export | Saves the final metrics in a machine-readable JSON file |

## Results at a glance

| Task | Success rate |
|---|---:|
| **Overall** | **81%** |
| Battery → hazard bin | 44% |
| Chemical bottle → hazard bin | 92% |
| Plastic bottle → recycle bin | 88% |
| Syringe → hazard bin | 100% |
| Random policy | 0% |

## Scope of the system

SafeSort-RL is a hybrid reinforcement-learning system:

- PPO predicts small Cartesian movements: `Δx`, `Δy`, and `Δz`.
- Genesis inverse kinematics converts those movements into Franka Panda joint targets.
- The task instruction is represented by a four-way symbolic vector.
- Grasp attachment and release are handled by a deterministic safety layer.
- All reported evaluation results come from simulation.

> **Showcase note:** The final video uses a deterministic, physically checked manipulation sequence so each phase is easy to follow on screen. It is a presentation of the system, not a replacement for the PPO evaluation reported above.

The final showcase also uses dynamic rigid-body proxy objects, runtime weld constraints, and recognizable visual skins for the battery, bottles, and syringe. Phase labels are updated only after the corresponding motion is complete.


## 1. Run settings

This cell controls which parts of the notebook will run. The default configuration is designed for a normal submission run: load the trained model, build the scene, create the final video, and avoid expensive retraining or repeated evaluation.

| Setting | Default | When to enable it |
|---|---:|---|
| `RUN_TRAINING` | `False` | Only when intentionally training a new model |
| `RUN_FULL_EVALUATION` | `False` | To rerun all 100 balanced evaluation episodes |
| `RUN_RANDOM_BASELINE` | `False` | To repeat the 50-episode random-action baseline |
| `RUN_INSTRUCTION_SWAP_PROBE` | `False` | To regenerate the same-scene command-grounding test |
| `RUN_AMD_SNAPSHOT` | `False` | To capture a fresh `rocm-smi` hardware snapshot |
| `RUN_BENCHMARK` | `False` | To measure simulation control steps per second |
| `RUN_VIDEO` | `True` | To generate the final realistic side-panel showcase |

For a regular rerun, leave the settings as they are. Training and full evaluation take much longer and are not required to view the final result.


In [1]:
RUN_TRAINING = False
RUN_FULL_EVALUATION = False
RUN_RANDOM_BASELINE = False
RUN_INSTRUCTION_SWAP_PROBE = False
RUN_AMD_SNAPSHOT = False
RUN_BENCHMARK = False
RUN_VIDEO = True

FULL_EVALUATION_EPISODES = 100
RANDOM_BASELINE_EPISODES = 50
INSTRUCTION_SWAP_STEPS = 45

print("Configuration")
print("  Training:", RUN_TRAINING)
print("  Full evaluation:", RUN_FULL_EVALUATION)
print("  Random baseline:", RUN_RANDOM_BASELINE)
print("  Instruction-swap probe:", RUN_INSTRUCTION_SWAP_PROBE)
print("  AMD snapshot:", RUN_AMD_SNAPSHOT)
print("  Benchmark:", RUN_BENCHMARK)
print("  Final video:", RUN_VIDEO)


Configuration
  Training: False
  Full evaluation: False
  Random baseline: False
  Instruction-swap probe: False
  AMD snapshot: False
  Benchmark: False
  Final video: True


## 2. Imports and ROCm environment check

This cell loads the libraries used throughout the notebook and confirms that the active kernel is the expected AMD ROCm environment.

The important checks are:

- PyTorch can access the GPU.
- The PyTorch build includes HIP/ROCm support.
- Genesis is version `1.3.1`.
- Stable-Baselines3 is available for PPO loading and evaluation.

It also creates the working output folders (`models/`, `logs/`, and `videos/`) when they are missing.

> Stop here if the GPU or HIP assertions fail. Continuing with the wrong kernel can lead to very slow execution or a Genesis initialization error.


In [2]:
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import gymnasium as gym
import imageio.v2 as imageio
import numpy as np
import torch
import torch.nn as nn

import genesis as gs

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("HIP:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable")
print("Genesis:", gs.__version__)
print(
    "Stable-Baselines3:",
    importlib.metadata.version("stable-baselines3"),
)

assert torch.cuda.is_available(), "ROCm GPU is not available."
assert torch.version.hip is not None, "This is not a ROCm PyTorch build."
assert gs.__version__.startswith("1.3.1"), (
    f"Expected Genesis 1.3.1, found {gs.__version__}"
)

Path("models").mkdir(exist_ok=True)
Path("models/checkpoints").mkdir(parents=True, exist_ok=True)
Path("logs").mkdir(exist_ok=True)
Path("videos").mkdir(exist_ok=True)


[I 08/06/26 07:48:52.500 909511] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


Python executable: /workspace/genesis-clean/bin/python
Python: 3.12.3
PyTorch: 2.8.0+rocm6.4
HIP: 6.4.43482-0f2d60242
GPU available: True
GPU: AMD Radeon Graphics
Genesis: 1.3.1
Stable-Baselines3: 2.9.0


## 3. Initialize Genesis on the AMD GPU

Genesis is initialized with the `gs.amdgpu` backend, 32-bit precision, and a fixed random seed for reproducibility.

Run this cell **once after each kernel restart**. Genesis keeps global simulator state, so repeatedly initializing it in the same session can cause an error. If that happens, restart the kernel and begin again from the top.


In [4]:
gs.init(
    backend=gs.amdgpu,
    seed=42,
    precision="32",
    logging_level="warning",
    performance_mode=False,
)

print("✓ Genesis initialized")
print("Genesis backend:", gs.backend)
print("Genesis device:", gs.device)


GenesisException: Genesis already initialized.

## 4. Small tensor and entity helpers

Genesis, PyTorch, and NumPy return values in slightly different formats. These helper functions keep the rest of the notebook readable by handling three common operations:

1. converting tensors to NumPy arrays,
2. flattening batched vectors into a single state vector, and
3. moving an entity while clearing its previous velocity.

They are deliberately simple because they are used throughout reset, evaluation, rendering, and the physical showcase.


In [5]:
def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def as_vector(value):
    array = to_numpy(value)

    if array.ndim >= 2:
        array = array[0]

    return np.asarray(array, dtype=np.float32)


def get_entity_position(entity):
    return as_vector(entity.get_pos())


def set_entity_position(entity, position):
    entity.set_pos(
        np.asarray(position, dtype=np.float32),
        zero_velocity=True,
    )


## 5. SafeSort Genesis environment

This is the main environment used by both PPO evaluation and the final demonstration. A single Genesis scene contains the Franka Panda, two destination bins, four task objects, and three cameras.

### Sorting tasks

| Instruction target | Destination |
|---|---|
| Battery | Hazard bin |
| Chemical bottle | Hazard bin |
| Plastic bottle | Recycle bin |
| Syringe | Hazard bin |

### Policy observation

The PPO policy receives a compact 23-value state vector rather than raw camera images.

| Observation component | Values |
|---|---:|
| Gripper position | 3 |
| Four object positions | 12 |
| Target-bin position | 3 |
| Instruction one-hot vector | 4 |
| Holding flag | 1 |
| **Total** | **23** |

### Policy action

The action is a three-value continuous Cartesian update:

```text
[Δx, Δy, Δz]
```

Genesis inverse kinematics turns the updated hand position into Franka joint commands.

### Why the controller is hybrid

The learned policy is responsible for reaching and transport direction. Grasp attachment and release are deterministic safety operations. This keeps the learning problem manageable while still allowing the policy to decide how to move toward the instructed object and destination.

### Two object representations

The environment keeps two related object sets:

- fixed multi-part objects used by the original RL task and realistic rendering,
- dynamic single-body showcase objects used for physically stable weld, lift, and release tests.

The final video overlays the recognizable multi-part appearance on the verified dynamic proxy without allowing the decorative pieces to disturb the physics.


In [6]:
class GenesisSafeSortEnv(gym.Env):
    """Genesis 1.3.1 SafeSort environment for one reusable GPU scene."""

    metadata = {"render_modes": ["rgb_array"]}

    def __init__(
        self,
        show_viewer=False,
        max_steps=120,
        control_substeps=3,
    ):
        super().__init__()

        self.max_steps = int(max_steps)
        self.control_substeps = int(control_substeps)

        self.object_types = [
            "battery",
            "chemical_bottle",
            "plastic_bottle",
            "syringe",
        ]

        self.instructions = {
            "Move the battery to the hazard bin.": (
                "battery",
                "hazard",
            ),
            "Move the chemical bottle to the hazard bin.": (
                "chemical_bottle",
                "hazard",
            ),
            "Move the plastic bottle to the recycle bin.": (
                "plastic_bottle",
                "recycle",
            ),
            "Move the syringe to the hazard bin.": (
                "syringe",
                "hazard",
            ),
        }

        self.instruction_to_index = {
            instruction: index
            for index, instruction in enumerate(self.instructions)
        }

        # Targeted curriculum: focus additional fine-tuning on the two
        # weakest tasks while keeping all four tasks represented.
        self.training_instruction_probabilities = np.array(
            [
                0.55,  # battery
                0.15,  # chemical bottle
                0.15,  # plastic bottle
                0.15,  # syringe
            ],
            dtype=np.float32,
        )

        self.grasp_thresholds = {
            "battery": 0.14,
            "chemical_bottle": 0.17,
            "plastic_bottle": 0.14,
            "syringe": 0.16,
        }

        self.carry_offsets = {
            "battery": 0.08,
            "chemical_bottle": 0.11,
            "plastic_bottle": 0.09,
            "syringe": 0.07,
        }

        self.task_reward_scales = {
            "battery": 1.40,
            "chemical_bottle": 1.00,
            "plastic_bottle": 1.00,
            "syringe": 1.00,
        }

        self.bin_positions = {
            "hazard": np.array(
                [0.72, 0.23, 0.12],
                dtype=np.float32,
            ),
            "recycle": np.array(
                [0.43, 0.23, 0.12],
                dtype=np.float32,
            ),
        }

        self.action_space = gym.spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(3,),
            dtype=np.float32,
        )

        self.observation_space = gym.spaces.Box(
            low=-2.0,
            high=2.0,
            shape=(23,),
            dtype=np.float32,
        )

        self.scene = gs.Scene(
            show_viewer=show_viewer,
            sim_options=gs.options.SimOptions(
                # Smaller timestep prevents unstable Franka constraint forces.
                dt=0.001,
            ),
            # Runtime welds physically attach the dynamic showcase object
            # to the Franka hand after the fingers close.
            rigid_options=gs.options.RigidOptions(
                max_dynamic_constraints=8,
            ),
            viewer_options=gs.options.ViewerOptions(
                camera_pos=(1.30, -1.25, 1.10),
                camera_lookat=(0.56, 0.05, 0.15),
                camera_fov=48,
            ),
        )

        self.plane = self.scene.add_entity(
            morph=gs.morphs.Plane(),
        )

        self.table = self.scene.add_entity(
            morph=gs.morphs.Box(
                size=(0.78, 0.72, 0.06),
                pos=(0.56, 0.02, 0.03),
                fixed=True,
            ),
            surface=gs.surfaces.Rough(
                color=(0.36, 0.39, 0.44),
            ),
        )

        # Genesis 1.3.1 in this environment does not accept
        # enable_neutral_collision as an MJCF argument.
        self.franka = self.scene.add_entity(
            morph=gs.morphs.MJCF(
                file="xml/franka_emika_panda/panda.xml",
            ),
        )

        self._create_bins()
        self.objects = self._create_realistic_objects()

        # A hidden parking shelf keeps unused dynamic showcase objects stable.
        # It sits outside every camera view.
        self.showcase_parking_shelf = self._add_box(
            size=(0.75, 0.45, 0.05),
            pos=(1.55, -0.65, 0.025),
            color=(0.05, 0.05, 0.06),
            fixed=True,
        )
        self.showcase_objects = self._create_dynamic_showcase_objects()

        # Evaluation/wide camera.
        self.camera = self.scene.add_camera(
            res=(704, 432),
            pos=(1.20, -1.05, 0.88),
            lookat=(0.56, 0.04, 0.17),
            fov=42,
            GUI=False,
        )

        # Close-up camera focused on the object and Franka fingers.
        self.grasp_camera = self.scene.add_camera(
            res=(704, 432),
            pos=(0.92, -0.68, 0.52),
            lookat=(0.53, -0.08, 0.18),
            fov=34,
            GUI=False,
        )

        # Placement camera focused on both destination bins.
        self.bin_camera = self.scene.add_camera(
            res=(704, 432),
            pos=(0.92, -0.42, 0.62),
            lookat=(0.56, 0.23, 0.17),
            fov=34,
            GUI=False,
        )

        print("Building Genesis GPU scene (one-time JIT compilation)...")
        print("Do not rerun this cell while it is active.")

        build_start = time.perf_counter()
        self.scene.build()
        self.scene_build_seconds = (
            time.perf_counter() - build_start
        )

        print(
            f"✓ Genesis GPU scene built in "
            f"{self.scene_build_seconds:.1f} seconds"
        )

        self.end_effector = self.franka.get_link("hand")
        self.arm_dofs = np.arange(7)
        self.finger_dofs = np.arange(7, 9)

        self.rigid_solver = self.scene.sim.rigid_solver
        if not hasattr(self.rigid_solver, "add_weld_constraint"):
            raise RuntimeError(
                "This Genesis build does not expose runtime weld constraints."
            )

        self.showcase_links = {
            object_type: self._single_rigid_link(entity)
            for object_type, entity in self.showcase_objects.items()
        }
        self.active_showcase_weld = None

        for link in self.showcase_links.values():
            try:
                link.set_mass(0.10)
            except Exception:
                pass
            try:
                link.set_friction(1.0)
            except Exception:
                pass

        self._configure_franka()

        # This must be initialized before _set_franka_pose_direct(),
        # because that method writes it into the two finger joint values.
        self.gripper_opening = 0.04

        self.gripper_position = np.array(
            [0.56, -0.05, 0.46],
            dtype=np.float32,
        )

        # Put the robot into its clean initial pose without controller motion.
        self._set_franka_pose_direct(
            self.gripper_position
        )

        # Save the complete clean state only after the robot and fingers
        # have been initialized successfully.
        self.initial_scene_state = self.scene.get_state()

        self.step_count = 0
        self.current_instruction = None
        self.target_object = None
        self.target_bin = None
        self.held_object = None

        self.previous_object_distance = 0.0
        self.previous_bin_distance = 0.0

    def _add_box(
        self,
        size,
        pos,
        color,
        fixed=True,
        rough=True,
    ):
        surface_class = (
            gs.surfaces.Rough
            if rough
            else gs.surfaces.Smooth
        )

        return self.scene.add_entity(
            morph=gs.morphs.Box(
                size=size,
                pos=pos,
                fixed=fixed,
            ),
            surface=surface_class(
                color=color,
            ),
        )

    def _add_cylinder(
        self,
        radius,
        height,
        pos,
        color,
        fixed=True,
        rough=False,
    ):
        surface_class = (
            gs.surfaces.Rough
            if rough
            else gs.surfaces.Smooth
        )

        return self.scene.add_entity(
            morph=gs.morphs.Cylinder(
                radius=radius,
                height=height,
                pos=pos,
                fixed=fixed,
            ),
            surface=surface_class(
                color=color,
            ),
        )

    def _create_bins(self):
        """Create large open-front red and green bins."""
        self.bin_parts = {
            "hazard": [],
            "recycle": [],
        }

        specifications = {
            "hazard": {
                "center": self.bin_positions["hazard"],
                "color": (0.95, 0.04, 0.04),
            },
            "recycle": {
                "center": self.bin_positions["recycle"],
                "color": (0.04, 0.85, 0.10),
            },
        }

        for bin_name, specification in specifications.items():
            center = specification["center"]
            color = specification["color"]

            # Base
            self.bin_parts[bin_name].append(
                self._add_box(
                    size=(0.22, 0.18, 0.04),
                    pos=(center[0], center[1], 0.08),
                    color=color,
                )
            )

            # Back wall
            self.bin_parts[bin_name].append(
                self._add_box(
                    size=(0.22, 0.025, 0.20),
                    pos=(
                        center[0],
                        center[1] + 0.09,
                        0.16,
                    ),
                    color=color,
                )
            )

            # Left wall
            self.bin_parts[bin_name].append(
                self._add_box(
                    size=(0.025, 0.18, 0.20),
                    pos=(
                        center[0] - 0.11,
                        center[1],
                        0.16,
                    ),
                    color=color,
                )
            )

            # Right wall
            self.bin_parts[bin_name].append(
                self._add_box(
                    size=(0.025, 0.18, 0.20),
                    pos=(
                        center[0] + 0.11,
                        center[1],
                        0.16,
                    ),
                    color=color,
                )
            )

    def _create_realistic_objects(self):
        """
        Create procedural multi-part objects.

        Each object group has a primary physics entity and decorative entities.
        All parts are fixed and moved together by the hybrid grasp controller.
        """
        objects = {}

        # Battery: yellow body, dark label, two metallic terminals.
        battery_primary = self._add_box(
            size=(0.085, 0.052, 0.115),
            pos=(0.48, -0.13, 0.125),
            color=(0.98, 0.78, 0.03),
        )
        battery_label = self._add_box(
            size=(0.060, 0.054, 0.045),
            pos=(0.48, -0.13, 0.125),
            color=(0.12, 0.12, 0.12),
        )
        battery_plus = self._add_cylinder(
            radius=0.009,
            height=0.014,
            pos=(0.46, -0.13, 0.190),
            color=(0.80, 0.80, 0.82),
        )
        battery_minus = self._add_cylinder(
            radius=0.007,
            height=0.011,
            pos=(0.50, -0.13, 0.188),
            color=(0.20, 0.20, 0.22),
        )

        objects["battery"] = {
            "primary": battery_primary,
            "parts": [
                battery_primary,
                battery_label,
                battery_plus,
                battery_minus,
            ],
            "offsets": [
                np.array([0.0, 0.0, 0.0]),
                np.array([0.0, 0.0, 0.0]),
                np.array([-0.02, 0.0, 0.065]),
                np.array([0.02, 0.0, 0.063]),
            ],
        }

        # Chemical bottle: red body, shoulder, neck, black cap, warning label.
        chemical_primary = self._add_cylinder(
            radius=0.038,
            height=0.115,
            pos=(0.59, -0.13, 0.130),
            color=(0.82, 0.05, 0.05),
            rough=True,
        )
        chemical_shoulder = self._add_cylinder(
            radius=0.030,
            height=0.025,
            pos=(0.59, -0.13, 0.200),
            color=(0.90, 0.07, 0.07),
            rough=True,
        )
        chemical_neck = self._add_cylinder(
            radius=0.017,
            height=0.032,
            pos=(0.59, -0.13, 0.228),
            color=(0.90, 0.07, 0.07),
            rough=True,
        )
        chemical_cap = self._add_cylinder(
            radius=0.021,
            height=0.018,
            pos=(0.59, -0.13, 0.253),
            color=(0.05, 0.05, 0.05),
            rough=True,
        )
        chemical_label = self._add_box(
            size=(0.050, 0.078, 0.040),
            pos=(0.59, -0.13, 0.130),
            color=(1.0, 0.90, 0.05),
        )

        objects["chemical_bottle"] = {
            "primary": chemical_primary,
            "parts": [
                chemical_primary,
                chemical_shoulder,
                chemical_neck,
                chemical_cap,
                chemical_label,
            ],
            "offsets": [
                np.array([0.0, 0.0, 0.0]),
                np.array([0.0, 0.0, 0.070]),
                np.array([0.0, 0.0, 0.098]),
                np.array([0.0, 0.0, 0.123]),
                np.array([0.0, 0.0, 0.0]),
            ],
        }

        # Plastic bottle: green body, shoulder, neck, blue cap, white label.
        plastic_primary = self._add_cylinder(
            radius=0.033,
            height=0.145,
            pos=(0.48, -0.01, 0.145),
            color=(0.10, 0.75, 0.22, 0.90),
        )
        plastic_shoulder = self._add_cylinder(
            radius=0.026,
            height=0.027,
            pos=(0.48, -0.01, 0.230),
            color=(0.12, 0.82, 0.26, 0.90),
        )
        plastic_neck = self._add_cylinder(
            radius=0.014,
            height=0.035,
            pos=(0.48, -0.01, 0.260),
            color=(0.12, 0.82, 0.26, 0.90),
        )
        plastic_cap = self._add_cylinder(
            radius=0.018,
            height=0.016,
            pos=(0.48, -0.01, 0.286),
            color=(0.04, 0.28, 0.90),
        )
        plastic_label = self._add_cylinder(
            radius=0.034,
            height=0.040,
            pos=(0.48, -0.01, 0.145),
            color=(0.92, 0.92, 0.92),
        )

        objects["plastic_bottle"] = {
            "primary": plastic_primary,
            "parts": [
                plastic_primary,
                plastic_shoulder,
                plastic_neck,
                plastic_cap,
                plastic_label,
            ],
            "offsets": [
                np.array([0.0, 0.0, 0.0]),
                np.array([0.0, 0.0, 0.085]),
                np.array([0.0, 0.0, 0.115]),
                np.array([0.0, 0.0, 0.141]),
                np.array([0.0, 0.0, 0.0]),
            ],
        }

        # Syringe: translucent barrel, blue plunger/flange, metallic needle.
        syringe_primary = self._add_cylinder(
            radius=0.017,
            height=0.110,
            pos=(0.64, -0.01, 0.145),
            color=(0.72, 0.86, 1.0, 0.75),
        )
        syringe_plunger = self._add_cylinder(
            radius=0.010,
            height=0.060,
            pos=(0.64, -0.01, 0.070),
            color=(0.14, 0.34, 0.95),
        )
        syringe_flange = self._add_box(
            size=(0.075, 0.020, 0.012),
            pos=(0.64, -0.01, 0.087),
            color=(0.16, 0.38, 0.95),
        )
        syringe_hub = self._add_cylinder(
            radius=0.009,
            height=0.020,
            pos=(0.64, -0.01, 0.210),
            color=(0.16, 0.40, 0.95),
        )
        syringe_needle = self._add_cylinder(
            radius=0.003,
            height=0.080,
            pos=(0.64, -0.01, 0.260),
            color=(0.78, 0.78, 0.82),
        )

        objects["syringe"] = {
            "primary": syringe_primary,
            "parts": [
                syringe_primary,
                syringe_plunger,
                syringe_flange,
                syringe_hub,
                syringe_needle,
            ],
            "offsets": [
                np.array([0.0, 0.0, 0.0]),
                np.array([0.0, 0.0, -0.075]),
                np.array([0.0, 0.0, -0.058]),
                np.array([0.0, 0.0, 0.065]),
                np.array([0.0, 0.0, 0.115]),
            ],
        }

        return objects

    def _create_dynamic_showcase_objects(self):
        """
        Create one physically simulated object per task.

        These objects are used only by the final showcase. The evaluated PPO
        environment keeps its original fixed multi-part objects unchanged.
        """
        parking = {
            "battery": (1.38, -0.65, 0.105),
            "chemical_bottle": (1.50, -0.65, 0.115),
            "plastic_bottle": (1.62, -0.65, 0.125),
            "syringe": (1.74, -0.65, 0.115),
        }

        objects = {
            "battery": self._add_box(
                size=(0.052, 0.034, 0.085),
                pos=parking["battery"],
                color=(0.98, 0.78, 0.03),
                fixed=False,
            ),
            "chemical_bottle": self._add_cylinder(
                radius=0.023,
                height=0.105,
                pos=parking["chemical_bottle"],
                color=(0.86, 0.05, 0.05),
                fixed=False,
                rough=True,
            ),
            "plastic_bottle": self._add_cylinder(
                radius=0.022,
                height=0.125,
                pos=parking["plastic_bottle"],
                color=(0.08, 0.76, 0.20, 0.95),
                fixed=False,
            ),
            "syringe": self._add_cylinder(
                radius=0.012,
                height=0.105,
                pos=parking["syringe"],
                color=(0.18, 0.45, 0.95, 0.95),
                fixed=False,
            ),
        }

        self.showcase_parking_positions = {
            key: np.asarray(value, dtype=np.float32)
            for key, value in parking.items()
        }

        return objects

    @staticmethod
    def _single_rigid_link(entity):
        links = list(entity.links)
        if len(links) != 1:
            raise RuntimeError(
                "A showcase object must contain exactly one rigid link."
            )
        return links[0]

    def _delete_showcase_weld(self):
        if getattr(self, "active_showcase_weld", None) is None:
            return

        link_object, link_hand = self.active_showcase_weld
        try:
            self.rigid_solver.delete_weld_constraint(
                link_object,
                link_hand,
            )
        finally:
            self.active_showcase_weld = None

    def add_showcase_weld(self, object_type):
        """Physically weld the active dynamic object to the Franka hand."""
        self._delete_showcase_weld()

        object_link = self.showcase_links[object_type]
        pair = (
            int(object_link.idx),
            int(self.end_effector.idx),
        )

        self.rigid_solver.add_weld_constraint(*pair)
        self.active_showcase_weld = pair
        self.scene.step()

    def release_showcase_weld(self):
        """Release the dynamic object so gravity and bin contacts take over."""
        self._delete_showcase_weld()
        self.scene.step()

    def park_showcase_objects(self):
        self._delete_showcase_weld()
        for object_type, entity in self.showcase_objects.items():
            entity.set_pos(
                self.showcase_parking_positions[object_type],
                zero_velocity=True,
            )
            entity.set_quat(
                np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32),
                zero_velocity=True,
            )

    def _configure_franka(self):
        """Use conservative gains for repeated IK control."""
        self.franka.set_dofs_kp(
            kp=np.array([800, 800, 700, 700, 450, 350, 250, 60, 60],dtype=np.float32)
        )

        self.franka.set_dofs_kv(
            kv=np.array(
                [80, 80, 70, 70, 45, 35, 25, 6, 6],
                dtype=np.float32,
            )
        )

        self.franka.set_dofs_force_range(
            lower=np.array(
                [
                    -87,
                    -87,
                    -87,
                    -87,
                    -12,
                    -12,
                    -12,
                    -40,
                    -40,
                ],
                dtype=np.float32,
            ),
            upper=np.array(
                [
                    87,
                    87,
                    87,
                    87,
                    12,
                    12,
                    12,
                    40,
                    40,
                ],
                dtype=np.float32,
            ),
        )

    def _sample_object_positions(self):
        """
        Sample separated object positions.

        Chemical bottle and syringe use a slightly safer curriculum region
        during targeted fine-tuning. A deterministic fallback guarantees that
        reset never fails.
        """
        positions = {}
        minimum_separation = 0.075

        fallback_positions = {
            "battery": np.array(
                [0.42, -0.17, 0.13],
                dtype=np.float32,
            ),
            "chemical_bottle": np.array(
                [0.55, -0.16, 0.13],
                dtype=np.float32,
            ),
            "plastic_bottle": np.array(
                [0.42, -0.04, 0.13],
                dtype=np.float32,
            ),
            "syringe": np.array(
                [0.55, -0.03, 0.13],
                dtype=np.float32,
            ),
        }

        for object_type in self.object_types:
            found_position = False

            for _ in range(1000):
                if object_type in {
                    "chemical_bottle",
                    "syringe",
                }:
                    x_low, x_high = 0.44, 0.65
                    y_low, y_high = -0.16, -0.01
                else:
                    x_low, x_high = 0.38, 0.72
                    y_low, y_high = -0.22, 0.04

                candidate = np.array(
                    [
                        self.np_random.uniform(
                            x_low,
                            x_high,
                        ),
                        self.np_random.uniform(
                            y_low,
                            y_high,
                        ),
                        0.13,
                    ],
                    dtype=np.float32,
                )

                separated = all(
                    np.linalg.norm(
                        candidate[:2]
                        - previous[:2]
                    )
                    >= minimum_separation
                    for previous in positions.values()
                )

                away_from_bins = all(
                    np.linalg.norm(
                        candidate[:2]
                        - bin_position[:2]
                    )
                    >= 0.14
                    for bin_position in self.bin_positions.values()
                )

                if separated and away_from_bins:
                    positions[object_type] = candidate
                    found_position = True
                    break

            if not found_position:
                print(
                    "Warning: random object placement failed. "
                    "Using deterministic fallback positions."
                )
                return fallback_positions

        return positions

    def _set_object_group_position(
        self,
        object_type,
        primary_position,
    ):
        group = self.objects[object_type]

        for part, offset in zip(
            group["parts"],
            group["offsets"],
        ):
            set_entity_position(
                part,
                np.asarray(
                    primary_position,
                    dtype=np.float32,
                )
                + offset,
            )

    def _solve_ik(self, target_position):
        qpos = self.franka.inverse_kinematics(
            link=self.end_effector,
            pos=np.asarray(
                target_position,
                dtype=np.float32,
            ),
            quat=np.array(
                [0.0, 1.0, 0.0, 0.0],
                dtype=np.float32,
            ),
        )

        qpos = to_numpy(qpos).astype(np.float32)

        if qpos.ndim > 1:
            qpos = qpos[0]

        if not np.all(np.isfinite(qpos)):
            raise RuntimeError("Genesis IK returned non-finite joint values.")

        return qpos

    def _set_franka_pose_direct(self, target_position):
        """Set a clean reset pose without dynamic controller motion."""
        qpos = self._solve_ik(target_position)
        qpos[-2:] = self.gripper_opening

        self.franka.set_qpos(
            qpos,
            zero_velocity=True,
        )

    def _move_franka_to(self, target_position):
        """
        Smoothly interpolate joint targets instead of applying one large IK
        jump. This substantially reduces constraint-force instability.
        """
        target_qpos = self._solve_ik(target_position)

        current_qpos = to_numpy(
            self.franka.get_qpos()
        ).astype(np.float32)

        if current_qpos.ndim > 1:
            current_qpos = current_qpos[0]

        if not np.all(np.isfinite(current_qpos)):
            raise RuntimeError(
                "Franka state contains non-finite joint values."
            )

        interpolation_steps = 3

        for alpha in np.linspace(
            0.2,
            1.0,
            interpolation_steps,
            dtype=np.float32,
        ):
            intermediate_qpos = (
                current_qpos[:7]
                + alpha
                * (
                    target_qpos[:7]
                    - current_qpos[:7]
                )
            )

            self.franka.control_dofs_position(
                position=intermediate_qpos,
                dofs_idx_local=self.arm_dofs,
            )

            self.franka.control_dofs_position(
                position=np.array(
                    [
                        self.gripper_opening,
                        self.gripper_opening,
                    ],
                    dtype=np.float32,
                ),
                dofs_idx_local=self.finger_dofs,
            )

            # One small physics step per interpolated waypoint.
            self.scene.step()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Remove any runtime showcase weld before restoring the checkpoint.
        self._delete_showcase_weld()

        # Restore robot, object, velocity, contact, and solver state.
        self.scene.reset(
            state=self.initial_scene_state,
        )

        self.step_count = 0
        self.held_object = None
        self.gripper_opening = 0.04

        sampled_positions = self._sample_object_positions()

        for object_type, position in sampled_positions.items():
            self._set_object_group_position(
                object_type,
                position,
            )

        if options and "instruction" in options:
            instruction = str(options["instruction"])

            if instruction not in self.instructions:
                raise ValueError(
                    f"Unknown instruction: {instruction}"
                )

            self.current_instruction = instruction
        else:
            instruction_list = list(
                self.instructions.keys()
            )

            self.current_instruction = str(
                self.np_random.choice(
                    instruction_list,
                    p=self.training_instruction_probabilities,
                )
            )

        self.target_object, self.target_bin = self.instructions[
            self.current_instruction
        ]

        self.gripper_position = np.array(
            [0.56, -0.05, 0.46],
            dtype=np.float32,
        )

        # Initialization should not generate controller impulses.
        self._set_franka_pose_direct(
            self.gripper_position
        )

        self.previous_object_distance = (
            self._distance_gripper_to_target()
        )
        self.previous_bin_distance = (
            self._distance_target_to_bin()
        )

        return (
            self._get_observation(),
            self._build_info(False),
        )

    def step(self, action):
        action = np.asarray(
            action,
            dtype=np.float32,
        )

        if action.shape != (3,):
            raise ValueError(
                f"Expected action shape (3,), got {action.shape}"
            )

        dx, dy, dz = np.clip(
            action,
            -1.0,
            1.0,
        )

        self.gripper_position = np.array(
            [
                np.clip(
                    self.gripper_position[0] + dx * 0.0075,
                    0.35,
                    0.80,
                ),
                np.clip(
                    self.gripper_position[1] + dy * 0.0075,
                    -0.27,
                    0.35,
                ),
                np.clip(
                    self.gripper_position[2] + dz * 0.0075,
                    0.16,
                    0.68,
                ),
            ],
            dtype=np.float32,
        )

        self._move_franka_to(
            self.gripper_position
        )

        grasped_this_step = False
        released_this_step = False

        if self.held_object is None:
            grasp_threshold = self.grasp_thresholds[
                self.target_object
            ]

            if (
                self._distance_gripper_to_target()
                < grasp_threshold
            ):
                self.held_object = self.target_object
                self.gripper_opening = 0.012
                grasped_this_step = True

        if self.held_object == self.target_object:
            carry_position = (
                self.gripper_position.copy()
            )
            carry_position[2] = max(
                0.13,
                carry_position[2]
                - self.carry_offsets[
                    self.target_object
                ],
            )

            self._set_object_group_position(
                self.target_object,
                carry_position,
            )

        object_distance = (
            self._distance_gripper_to_target()
        )
        bin_distance = (
            self._distance_target_to_bin()
        )

        if (
            self.held_object == self.target_object
            and bin_distance < 0.10
        ):
            self.held_object = None
            self.gripper_opening = 0.04
            released_this_step = True

        success = bool(
            released_this_step
            and bin_distance < 0.13
        )

        reward = -0.01

        if (
            not grasped_this_step
            and self.held_object is None
            and not success
        ):
            reward += 8.0 * (
                self.previous_object_distance
                - object_distance
            )

        if grasped_this_step:
            reward += 8.0

        if self.held_object == self.target_object:
            reward += 10.0 * (
                self.previous_bin_distance
                - bin_distance
            )

        if success:
            reward += 30.0

        # Give the weak tasks a larger learning signal during fine-tuning.
        reward *= self.task_reward_scales[
            self.target_object
        ]

        self.previous_object_distance = object_distance
        self.previous_bin_distance = bin_distance
        self.step_count += 1

        terminated = success
        truncated = self.step_count >= self.max_steps

        info = self._build_info(success)
        info.update(
            {
                "grasped_this_step": grasped_this_step,
                "released_this_step": released_this_step,
            }
        )

        return (
            self._get_observation(),
            float(reward),
            terminated,
            truncated,
            info,
        )

    def _target_position(self):
        return get_entity_position(
            self.objects[
                self.target_object
            ]["primary"]
        )

    def _distance_gripper_to_target(self):
        return float(
            np.linalg.norm(
                self.gripper_position
                - self._target_position()
            )
        )

    def _distance_target_to_bin(self):
        return float(
            np.linalg.norm(
                self._target_position()[:2]
                - self.bin_positions[
                    self.target_bin
                ][:2]
            )
        )

    def _get_observation(self):
        values = list(
            self.gripper_position
        )

        for object_type in self.object_types:
            values.extend(
                get_entity_position(
                    self.objects[
                        object_type
                    ]["primary"]
                )
            )

        values.extend(
            self.bin_positions[
                self.target_bin
            ]
        )

        instruction_vector = np.zeros(
            len(self.instructions),
            dtype=np.float32,
        )

        instruction_vector[
            self.instruction_to_index[
                self.current_instruction
            ]
        ] = 1.0

        holding_flag = np.array(
            [
                1.0
                if self.held_object == self.target_object
                else 0.0
            ],
            dtype=np.float32,
        )

        return np.concatenate(
            [
                np.asarray(
                    values,
                    dtype=np.float32,
                ),
                instruction_vector,
                holding_flag,
            ]
        ).astype(np.float32)

    def _build_info(self, success):
        return {
            "success": bool(success),
            "instruction": self.current_instruction,
            "target_object": self.target_object,
            "target_bin": self.target_bin,
            "held_object": self.held_object,
            "object_distance": (
                self._distance_gripper_to_target()
            ),
            "bin_distance": (
                self._distance_target_to_bin()
            ),
        }

    def set_demo_gripper_opening(self, opening, settle_steps=8):
        """Visually open or close Franka fingers for demo clarity."""
        self.gripper_opening = float(
            np.clip(opening, 0.008, 0.04)
        )

        self.franka.control_dofs_position(
            position=np.array(
                [self.gripper_opening, self.gripper_opening],
                dtype=np.float32,
            ),
            dofs_idx_local=self.finger_dofs,
        )

        for _ in range(int(settle_steps)):
            self.scene.step()

    def _render_camera(self, camera):
        rgb, _, _, _ = camera.render(
            rgb=True,
            depth=False,
            segmentation=False,
            normal=False,
            force_render=True,
        )

        frame = to_numpy(rgb)

        if frame.ndim == 4:
            frame = frame[0]

        if frame.shape[-1] == 4:
            frame = frame[..., :3]

        if frame.dtype != np.uint8:
            if frame.max() <= 1.0:
                frame = frame * 255.0

            frame = np.clip(
                frame,
                0,
                255,
            ).astype(np.uint8)

        return frame

    def render(self, view="wide"):
        cameras = {
            "wide": self.camera,
            "grasp": self.grasp_camera,
            "bin": self.bin_camera,
        }

        if view not in cameras:
            raise ValueError(
                f"Unknown camera view: {view}. "
                f"Expected one of {tuple(cameras)}."
            )

        return self._render_camera(
            cameras[view]
        )

    def close(self):
        # Keep the single Genesis scene alive for reuse.
        pass


print("✓ GenesisSafeSortEnv defined")


✓ GenesisSafeSortEnv defined


## 6. Build one reusable GPU scene

Creating the environment triggers the one-time Genesis scene build and GPU compilation. This can take a little while on the first run.

After the cell prints that the scene is ready, keep using the same `base_env` instance for the smoke test, evaluation, probes, and video generation. Rebuilding the scene is unnecessary and can consume additional GPU memory.


In [7]:
scene_build_start = time.perf_counter()

base_env = GenesisSafeSortEnv(
    show_viewer=False,
    max_steps=120,
    control_substeps=1,
)

scene_build_elapsed = (
    time.perf_counter()
    - scene_build_start
)

print("✓ SafeSort Genesis scene ready")
print(
    f"Environment creation time: "
    f"{scene_build_elapsed:.1f} seconds"
)


[Genesis] [07:49:45] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint1`
[Genesis] [07:49:45] [WARNING] (MJCF) Approximating tendon by joint actuator for `finger_joint2`
Building Genesis GPU scene (one-time JIT compilation)...
Do not rerun this cell while it is active.
[Genesis] [07:49:46] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [07:49:48] [WARNING] Filtered out geometry pairs causing self-collision for the neutral configuration (qpos0): (7, 12), (8, 12), (15, 22), (16, 21), (17, 24), (18, 23). Consider tuning Morph option 'decompose_robot_error_threshold' or specify dedicated collision meshes. This behavior can be disabled by setting Morph option 'enable_neutral_collision=True'.
✓ Genesis GPU scene built in 16.3 seconds
✓ SafeSort Genesis scene ready
Environment creation time: 21.1 seconds


## 7. Quick smoke test

Before loading the model, this short test checks that the environment can reset cleanly and produce a valid observation.

A successful run confirms that:

- the observation has shape `(23,)`,
- every observation value is finite,
- the instruction maps to the expected object and bin, and
- the initial distance values can be computed.

This is a useful stopping point for debugging. If the smoke test fails, fix the environment before running evaluation or the final video.


In [7]:
smoke_instruction = (
    "Move the battery to the hazard bin."
)

obs, info = base_env.reset(
    seed=12345,
    options={
        "instruction": smoke_instruction,
    },
)

print("Observation shape:", obs.shape)
print("Target object:", info["target_object"])
print("Target bin:", info["target_bin"])
print("Initial object distance:", info["object_distance"])
print("Initial bin distance:", info["bin_distance"])

assert obs.shape == (23,)
assert np.all(np.isfinite(obs))

print("✓ Environment smoke test passed")


Observation shape: (23,)
Target object: battery
Target bin: hazard
Initial object distance: 0.3565526008605957
Initial bin distance: 0.45185789465904236
✓ Environment smoke test passed


## 8. Training history

The final model was not produced in one uninterrupted run. Training was adjusted in stages after looking at per-task performance.

| Stage | Overall | Battery | Chemical | Plastic | Syringe | What changed |
|---|---:|---:|---:|---:|---:|---|
| Stable baseline | 61% | 76% | 28% | 84% | 56% | Equal exposure to all four tasks |
| Targeted fine-tuning | 76% | 24% | 88% | 92% | 100% | More chemical and syringe episodes |
| Balanced recovery | **81%** | **44%** | **92%** | **88%** | **100%** | Reintroduced battery-focused training without losing the stronger tasks |

The targeted stage fixed the weakest tasks, but it also showed a clear case of catastrophic forgetting on the battery task. A shorter balanced recovery stage was then used to regain part of the battery performance while preserving the other improvements.

### PPO settings used for the training runs

| Parameter | Value |
|---|---:|
| Policy | `MlpPolicy` |
| Learning rate | `3e-4` |
| Rollout length (`n_steps`) | `1024` |
| Batch size | `256` |
| Epochs per update | `10` |
| Discount factor (`gamma`) | `0.995` |
| GAE lambda | `0.95` |
| Clip range | `0.2` |
| Entropy coefficient | `0.005` |
| Policy/value network | `[256, 256]` |
| PPO inference device | CPU |
| Genesis simulation device | AMD GPU through ROCm |

The checkpoint-loading section below prefers `safesort_genesis_amd_final_best.zip`, then falls back to the balanced, targeted, or stable checkpoint when needed.


In [8]:
TRAINING_HISTORY = {
    "stable": {
        "overall": 0.61,
        "battery": 0.76,
        "chemical_bottle": 0.28,
        "plastic_bottle": 0.84,
        "syringe": 0.56,
    },
    "targeted": {
        "overall": 0.76,
        "battery": 0.24,
        "chemical_bottle": 0.88,
        "plastic_bottle": 0.92,
        "syringe": 1.00,
    },
    "balanced_final": {
        "overall": 0.81,
        "battery": 0.44,
        "chemical_bottle": 0.92,
        "plastic_bottle": 0.88,
        "syringe": 1.00,
    },
}

print(json.dumps(TRAINING_HISTORY, indent=2))


{
  "stable": {
    "overall": 0.61,
    "battery": 0.76,
    "chemical_bottle": 0.28,
    "plastic_bottle": 0.84,
    "syringe": 0.56
  },
  "targeted": {
    "overall": 0.76,
    "battery": 0.24,
    "chemical_bottle": 0.88,
    "plastic_bottle": 0.92,
    "syringe": 1.0
  },
  "balanced_final": {
    "overall": 0.81,
    "battery": 0.44,
    "chemical_bottle": 0.92,
    "plastic_bottle": 0.88,
    "syringe": 1.0
  }
}


## 9. Optional training function

The training function is included so the notebook documents how a fresh PPO run can be created. It saves periodic checkpoints under `models/checkpoints/` and writes the final model to a new filename.

Training is disabled by default. This is intentional: opening the final submission notebook should not start a long experiment or overwrite the model used for the reported results.

Enable `RUN_TRAINING` only when a new training run is actually required.


In [9]:
def train_new_policy(
    env,
    output_path=(
        "models/"
        "safesort_genesis_new_run"
    ),
    total_timesteps=100_000,
    device="cpu",
):
    monitored_env = Monitor(env)
    vector_env = DummyVecEnv(
        [lambda: monitored_env]
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=25_000,
        save_path="models/checkpoints",
        name_prefix="safesort_genesis",
    )

    policy_kwargs = {
        "activation_fn": nn.ReLU,
        "ortho_init": False,
        "net_arch": {
            "pi": [256, 256],
            "vf": [256, 256],
        },
    }

    model = PPO(
        "MlpPolicy",
        vector_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        device=device,
        verbose=1,
        tensorboard_log="./logs/",
        policy_kwargs=policy_kwargs,
        seed=42,
    )

    model.learn(
        total_timesteps=total_timesteps,
        callback=checkpoint_callback,
        progress_bar=False,
    )

    model.save(output_path)
    print(
        f"✓ Saved trained policy to "
        f"{output_path}.zip"
    )

    return model


trained_from_scratch_model = None

if RUN_TRAINING:
    trained_from_scratch_model = (
        train_new_policy(
            env=base_env,
            total_timesteps=100_000,
            device="cpu",
        )
    )
else:
    print(
        "Training skipped. "
        "The final checkpoint will be loaded."
    )


Training skipped. The final checkpoint will be loaded.


## 10. Load the trained PPO checkpoint

The notebook searches the `models/` folder from the most refined checkpoint to the oldest fallback:

1. `safesort_genesis_amd_final_best.zip`
2. `safesort_genesis_amd_balanced.zip`
3. `safesort_genesis_amd_targeted.zip`
4. `safesort_genesis_amd_stable.zip`

The first file found is loaded on the CPU for PPO inference. Genesis simulation remains on the AMD GPU.

If no checkpoint is found, the notebook still builds the environment and exports hardware information, but model-dependent evaluation and video cells are skipped.


In [10]:
MODEL_CANDIDATES = [
    "models/safesort_genesis_amd_final_best",
    "models/safesort_genesis_amd_balanced",
    "models/safesort_genesis_amd_targeted",
    "models/safesort_genesis_amd_stable",
]

selected_model_path = next(
    (
        path
        for path in MODEL_CANDIDATES
        if Path(path + ".zip").exists()
    ),
    None,
)

final_model = None

if selected_model_path is None:
    print("⚠ No trained checkpoint was found.")
    print("Copy one of these files into the models/ folder:")
    for path in MODEL_CANDIDATES:
        print(" ", path + ".zip")
    print("Model-dependent evaluation and video cells will be skipped.")
else:
    final_model = PPO.load(
        selected_model_path,
        device="cpu",
    )

    print("✓ Loaded:", selected_model_path + ".zip")
    print("Policy device:", final_model.device)
    print("Saved timesteps:", final_model.num_timesteps)
    print("Genesis simulation device:", gs.device)


✓ Loaded: models/safesort_genesis_amd_final_best.zip
Policy device: cpu
Saved timesteps: 138468
Genesis simulation device: cuda:0


## 11. Record the AMD / ROCm environment

This section saves the software and hardware details used for the run. The information is written in both JSON and plain-text formats:

```text
results/amd_environment.json
results/amd_environment.txt
```

The recorded fields include Python, PyTorch, HIP, GPU name, Genesis version, Genesis backend/device, and Stable-Baselines3 version.

A deeper `rocm-smi` snapshot is optional. Set `RUN_AMD_SNAPSHOT=True` when current GPU utilization, memory, power, clocks, and temperature are needed for evidence or benchmarking.


In [11]:
def capture_amd_environment(
    output_directory="results",
):
    output_directory = Path(output_directory)
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    environment = {
        "python": sys.version,
        "pytorch": torch.__version__,
        "hip": torch.version.hip,
        "gpu_available": bool(
            torch.cuda.is_available()
        ),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "genesis": gs.__version__,
        "genesis_backend": str(gs.backend),
        "genesis_device": str(gs.device),
        "stable_baselines3": (
            importlib.metadata.version(
                "stable-baselines3"
            )
        ),
    }

    json_path = (
        output_directory
        / "amd_environment.json"
    )

    json_path.write_text(
        json.dumps(
            environment,
            indent=2,
        ),
        encoding="utf-8",
    )

    text_lines = [
        f"{key}: {value}"
        for key, value in environment.items()
    ]

    text_path = (
        output_directory
        / "amd_environment.txt"
    )

    text_path.write_text(
        "\n".join(text_lines),
        encoding="utf-8",
    )

    print(
        json.dumps(
            environment,
            indent=2,
        )
    )
    print("✓ Saved:", json_path)
    print("✓ Saved:", text_path)

    return environment


def capture_rocm_snapshot(
    output_directory="results",
):
    output_directory = Path(output_directory)
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    rocm_smi = shutil.which("rocm-smi")

    if rocm_smi is None:
        print(
            "⚠ rocm-smi is unavailable in "
            "this notebook process."
        )
        return None

    command = [
        rocm_smi,
        "--showuse",
        "--showmemuse",
        "--showpower",
        "--showclocks",
        "--showtemp",
        "--json",
    ]

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    snapshot_path = (
        output_directory
        / "rocm_snapshot.json"
    )

    if completed.returncode != 0:
        print(
            "⚠ rocm-smi returned an error:"
        )
        print(completed.stderr)
        return None

    snapshot_path.write_text(
        completed.stdout,
        encoding="utf-8",
    )

    print("✓ Saved:", snapshot_path)
    print(completed.stdout[:2500])

    return completed.stdout


amd_environment = capture_amd_environment()

if RUN_AMD_SNAPSHOT:
    rocm_snapshot = capture_rocm_snapshot()
else:
    rocm_snapshot = None
    print(
        "Set RUN_AMD_SNAPSHOT=True to "
        "capture a fresh rocm-smi snapshot."
    )


{
  "python": "3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]",
  "pytorch": "2.8.0+rocm6.4",
  "hip": "6.4.43482-0f2d60242",
  "gpu_available": true,
  "gpu": "AMD Radeon Graphics",
  "genesis": "1.3.1",
  "genesis_backend": "3",
  "genesis_device": "cuda:0",
  "stable_baselines3": "2.9.0"
}
✓ Saved: results/amd_environment.json
✓ Saved: results/amd_environment.txt
Set RUN_AMD_SNAPSHOT=True to capture a fresh rocm-smi snapshot.


## 12. Balanced PPO evaluation

The evaluation cycles evenly through the four instructions, uses deterministic PPO actions, and records success, reward, episode length, and final object-to-bin distance.

When `RUN_FULL_EVALUATION=True`, the notebook performs 100 fresh episodes. With four tasks, this gives 25 episodes per instruction.

The full evaluation is disabled by default because it is time-consuming. In that case, the notebook uses the previously verified final measurements:

| Metric | Verified value |
|---|---:|
| Overall success | 81% |
| Mean reward | 38.930 |
| Reward standard deviation | 19.871 |
| Mean episode length | 70.1 steps |
| Mean final bin distance | 0.158 m |

Keeping the saved values and the rerun option separate makes it clear whether a result came from the recorded final experiment or from a fresh execution of this notebook.


In [12]:
def evaluate_policy(
    env,
    model,
    n_episodes=100,
    seed=20_000,
):
    instructions = list(
        env.instructions.keys()
    )

    successes = 0
    rewards = []
    lengths = []
    final_distances = []

    per_task = {
        instruction: {
            "successes": 0,
            "episodes": 0,
        }
        for instruction in instructions
    }

    for episode in range(n_episodes):
        instruction = instructions[
            episode % len(instructions)
        ]

        obs, info = env.reset(
            seed=seed + episode,
            options={"instruction": instruction},
        )

        episode_reward = 0.0

        for step_index in range(env.max_steps):
            action, _ = model.predict(
                obs,
                deterministic=True,
            )

            obs, reward, terminated, truncated, info = (
                env.step(action)
            )

            episode_reward += float(reward)

            if terminated or truncated:
                success = int(
                    info["success"]
                )

                successes += success

                per_task[instruction][
                    "successes"
                ] += success

                per_task[instruction][
                    "episodes"
                ] += 1

                lengths.append(
                    step_index + 1
                )

                final_distances.append(
                    info["bin_distance"]
                )
                break

        rewards.append(
            episode_reward
        )

    success_rate = (
        successes / n_episodes
    )

    print(
        f"Overall success rate: "
        f"{success_rate:.1%}"
    )
    print(
        f"Mean reward: "
        f"{np.mean(rewards):.3f}"
    )
    print(
        f"Reward std: "
        f"{np.std(rewards):.3f}"
    )
    print(
        f"Mean episode length: "
        f"{np.mean(lengths):.1f}"
    )
    print(
        f"Mean final bin distance: "
        f"{np.mean(final_distances):.3f}"
    )

    print("\nPer-task success:")

    for instruction, result in per_task.items():
        rate = (
            result["successes"]
            / result["episodes"]
        )

        print(
            f"  {instruction}: "
            f"{rate:.1%}"
        )

    return {
        "success_rate": success_rate,
        "mean_reward": float(
            np.mean(rewards)
        ),
        "reward_std": float(
            np.std(rewards)
        ),
        "mean_episode_length": float(
            np.mean(lengths)
        ),
        "mean_final_bin_distance": float(
            np.mean(final_distances)
        ),
        "per_task": per_task,
    }


In [13]:
VERIFIED_FINAL_RESULTS = {
    "success_rate": 0.81,
    "mean_reward": 38.930,
    "reward_std": 19.871,
    "mean_episode_length": 70.1,
    "mean_final_bin_distance": 0.158,
    "per_task_success": {
        "Move the battery to the hazard bin.": 0.44,
        "Move the chemical bottle to the hazard bin.": 0.92,
        "Move the plastic bottle to the recycle bin.": 0.88,
        "Move the syringe to the hazard bin.": 1.00,
    },
}

if RUN_FULL_EVALUATION:
    if final_model is None:
        print(
            "⚠ Full evaluation skipped because "
            "no model checkpoint is loaded."
        )
        final_results = (
            VERIFIED_FINAL_RESULTS.copy()
        )
        evaluation_runtime = None
    else:
        evaluation_start = (
            time.perf_counter()
        )

        final_results = evaluate_policy(
            env=base_env,
            model=final_model,
            n_episodes=FULL_EVALUATION_EPISODES,
            seed=150_000,
        )

        evaluation_runtime = (
            time.perf_counter()
            - evaluation_start
        )

        approximate_steps = (
            FULL_EVALUATION_EPISODES
            * final_results[
                "mean_episode_length"
            ]
        )

        final_results[
            "evaluation_wall_seconds"
        ] = float(evaluation_runtime)

        final_results[
            "episodes_per_second"
        ] = float(
            FULL_EVALUATION_EPISODES
            / evaluation_runtime
        )

        final_results[
            "approx_steps_per_second"
        ] = float(
            approximate_steps
            / evaluation_runtime
        )

        print(
            f"Evaluation wall time: "
            f"{evaluation_runtime:.1f}s"
        )
        print(
            f"Episodes/sec: "
            f"{final_results['episodes_per_second']:.3f}"
        )
        print(
            f"Approx. environment steps/sec: "
            f"{final_results['approx_steps_per_second']:.2f}"
        )
else:
    final_results = (
        VERIFIED_FINAL_RESULTS.copy()
    )
    evaluation_runtime = None

    print(
        "Loaded verified final evaluation "
        "results."
    )
    print(
        json.dumps(
            final_results,
            indent=2,
        )
    )


Loaded verified final evaluation results.
{
  "success_rate": 0.81,
  "mean_reward": 38.93,
  "reward_std": 19.871,
  "mean_episode_length": 70.1,
  "mean_final_bin_distance": 0.158,
  "per_task_success": {
    "Move the battery to the hazard bin.": 0.44,
    "Move the chemical bottle to the hazard bin.": 0.92,
    "Move the plastic bottle to the recycle bin.": 0.88,
    "Move the syringe to the hazard bin.": 1.0
  }
}


## 13. Random-policy baseline

A learned policy is more meaningful when it is compared with a simple baseline. Here, the environment is run with uniformly sampled actions from the same action space.

The verified random-policy result was:

- **0% success**
- **−1.509 mean reward**

Set `RUN_RANDOM_BASELINE=True` to repeat the baseline for 50 episodes. It is disabled during a normal submission run to save time.


In [14]:
def evaluate_random_policy(
    env,
    n_episodes=50,
    seed=40_000,
):
    successes = 0
    rewards = []

    for episode in range(n_episodes):
        obs, info = env.reset(
            seed=seed + episode,
        )

        episode_reward = 0.0

        for _ in range(env.max_steps):
            action = env.action_space.sample()

            obs, reward, terminated, truncated, info = (
                env.step(action)
            )

            episode_reward += float(reward)

            if terminated or truncated:
                successes += int(
                    info["success"]
                )
                break

        rewards.append(
            episode_reward
        )

    print(
        f"Random-policy success: "
        f"{successes / n_episodes:.1%}"
    )
    print(
        f"Random-policy mean reward: "
        f"{np.mean(rewards):.3f}"
    )

    return {
        "success_rate": successes / n_episodes,
        "mean_reward": float(np.mean(rewards)),
    }


In [15]:
if RUN_RANDOM_BASELINE:
    random_results = evaluate_random_policy(
        env=base_env,
        n_episodes=RANDOM_BASELINE_EPISODES,
        seed=180_000,
    )
else:
    random_results = {
        "success_rate": 0.0,
        "mean_reward": -1.509,
    }
    print("Verified random-policy success: 0.0%")
    print("Verified random-policy mean reward: -1.509")


Verified random-policy success: 0.0%
Verified random-policy mean reward: -1.509


## 14. Same-scene instruction-swap probe

This probe keeps the random seed and physical layout fixed, then changes only the instruction. The purpose is to check whether the model responds to the task vector rather than following one fixed object-selection routine.

For each instruction, the probe records the selected target and the change in distance to that target over a short rollout.

This is a grounding sanity check, not a separate success-rate evaluation. It shows whether the command influences behavior under the same scene conditions.


In [16]:
def run_instruction_swap_probe(
    env,
    model,
    seed=424_242,
    max_probe_steps=45,
):
    if model is None:
        raise RuntimeError(
            "A trained model must be loaded "
            "before running the probe."
        )

    instructions = list(
        env.instructions.keys()
    )

    results = []

    for instruction in instructions:
        obs, info = env.reset(
            seed=seed,
            options={
                "instruction": instruction,
            },
        )

        initial_positions = {
            object_type: (
                get_entity_position(
                    env.objects[
                        object_type
                    ]["primary"]
                ).copy()
            )
            for object_type in env.object_types
        }

        initial_distance = (
            env._distance_gripper_to_target()
        )

        minimum_distance = initial_distance
        grasped = False

        for step_index in range(
            max_probe_steps
        ):
            action, _ = model.predict(
                obs,
                deterministic=True,
            )

            (
                obs,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(action)

            minimum_distance = min(
                minimum_distance,
                info["object_distance"],
            )

            grasped = (
                info["held_object"]
                == info["target_object"]
            )

            if (
                grasped
                or terminated
                or truncated
            ):
                break

        results.append(
            {
                "instruction": instruction,
                "selected_target": (
                    info["target_object"]
                ),
                "same_scene_seed": seed,
                "initial_object_positions": {
                    key: value.tolist()
                    for key, value
                    in initial_positions.items()
                },
                "initial_distance": float(
                    initial_distance
                ),
                "minimum_distance": float(
                    minimum_distance
                ),
                "distance_reduction": float(
                    initial_distance
                    - minimum_distance
                ),
                "grasped": bool(grasped),
                "probe_steps": int(
                    step_index + 1
                ),
            }
        )

    return results


def print_instruction_swap_results(
    results,
):
    print(
        "Instruction-swap grounding probe"
    )
    print(
        "Same seed and scene; only the "
        "instruction changes.\n"
    )

    for result in results:
        print(
            "Instruction:",
            result["instruction"],
        )
        print(
            "Selected target:",
            result["selected_target"],
        )
        print(
            "Distance reduction:",
            f"{result['distance_reduction']:.3f}",
        )
        print(
            "Grasped:",
            result["grasped"],
        )
        print("-" * 52)


if RUN_INSTRUCTION_SWAP_PROBE:
    if final_model is None:
        instruction_swap_results = []
        print(
            "⚠ Probe skipped because no "
            "checkpoint is loaded."
        )
    else:
        instruction_swap_results = (
            run_instruction_swap_probe(
                env=base_env,
                model=final_model,
                seed=424_242,
                max_probe_steps=(
                    INSTRUCTION_SWAP_STEPS
                ),
            )
        )

        print_instruction_swap_results(
            instruction_swap_results
        )

        Path("results").mkdir(
            exist_ok=True
        )

        Path(
            "results/"
            "instruction_swap_probe.json"
        ).write_text(
            json.dumps(
                instruction_swap_results,
                indent=2,
            ),
            encoding="utf-8",
        )

        print(
            "✓ Saved: results/"
            "instruction_swap_probe.json"
        )
else:
    instruction_swap_results = []
    print(
        "Set RUN_INSTRUCTION_SWAP_PROBE=True "
        "to run the same-scene grounding probe."
    )


Set RUN_INSTRUCTION_SWAP_PROBE=True to run the same-scene grounding probe.


## 15. Simple command normalization

The trained policy expects one of four exact task instructions. This helper accepts a few more natural user phrases and maps them to the corresponding supported command.

Examples include:

| User wording | Normalized task |
|---|---|
| “Dispose of the battery safely.” | Battery → hazard bin |
| “Put the chemical container in hazardous waste.” | Chemical bottle → hazard bin |
| “Recycle this plastic bottle.” | Plastic bottle → recycle bin |
| “Handle the used syringe safely.” | Syringe → hazard bin |

This is a rule-based interface, not a language model. Unsupported wording raises a clear error instead of guessing.


In [17]:
def normalize_user_command(user_command):
    command = user_command.lower().strip()

    if "battery" in command:
        return "Move the battery to the hazard bin."

    if (
        "chemical" in command
        or "toxic bottle" in command
    ):
        return (
            "Move the chemical bottle "
            "to the hazard bin."
        )

    if (
        "plastic" in command
        or "recycle" in command
    ):
        return (
            "Move the plastic bottle "
            "to the recycle bin."
        )

    if (
        "syringe" in command
        or "needle" in command
    ):
        return "Move the syringe to the hazard bin."

    raise ValueError(
        "Unsupported command. Mention battery, "
        "chemical, plastic/recycle, syringe, or needle."
    )


example_commands = [
    "Dispose of the battery safely.",
    "Put the chemical container in hazardous waste.",
    "Recycle this plastic bottle.",
    "Handle the used syringe safely.",
]

for command in example_commands:
    print(
        command,
        "->",
        normalize_user_command(command),
    )


Dispose of the battery safely. -> Move the battery to the hazard bin.
Put the chemical container in hazardous waste. -> Move the chemical bottle to the hazard bin.
Recycle this plastic bottle. -> Move the plastic bottle to the recycle bin.
Handle the used syringe safely. -> Move the syringe to the hazard bin.


## 16. Optional simulation benchmark

The benchmark runs 200 environment control steps and reports approximate control steps per second together with the active Genesis device and ROCm information.

It is useful when comparing runtime settings or documenting AMD performance, but it is not part of the policy-quality evaluation. Leave `RUN_BENCHMARK=False` for the standard notebook run.


In [18]:
if RUN_BENCHMARK:
    obs, info = base_env.reset(seed=300_000)
    benchmark_steps = 200
    start = time.perf_counter()

    for index in range(benchmark_steps):
        obs, _, terminated, truncated, _ = base_env.step(
            base_env.action_space.sample()
        )

        if terminated or truncated:
            obs, info = base_env.reset(
                seed=300_001 + index
            )

    elapsed = time.perf_counter() - start

    print(
        "Control steps/sec:",
        benchmark_steps / elapsed,
    )
    print("Genesis backend:", gs.backend)
    print("Genesis device:", gs.device)
    print("HIP:", torch.version.hip)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "Benchmark disabled. Set RUN_BENCHMARK=True "
        "in the configuration cell to run it."
    )


Benchmark disabled. Set RUN_BENCHMARK=True in the configuration cell to run it.


## 17. Final four-object showcase

This section creates the presentation video after the environment and model checkpoint are ready.

### Output file

```text
videos/safesort_final_realistic_sidepanel_showcase.mp4
```

### What the video shows

Each task follows the same visible sequence:

1. introduce the instruction and destination,
2. approach the selected object,
3. align and close the fingers,
4. verify that the physical weld carries the object,
5. lift and transport it above the correct bin,
6. lower it to a safe, non-penetrating release pose,
7. release and clear the object,
8. retreat the arm, and
9. confirm that the object remains inside the destination bin.

### Presentation details

- A side panel displays the instruction, target, destination, current phase, and progress.
- The battery, chemical bottle, plastic bottle, and syringe use recognizable multi-part visual skins.
- The decorative skin is synchronized with the dynamic proxy only during rendering, so it does not interfere with contact or weld physics.
- Wide, grasp, and bin cameras are used where they make the action easiest to understand.

> The showcase sequence is deterministic and physically validated for visual clarity. The reported 81% success rate still comes from the PPO evaluation section, not from this scripted demonstration.


In [19]:
def wrap_text(text, max_characters=31):
    words = text.split()
    lines = []
    current = []

    for word in words:
        candidate = " ".join(current + [word])
        if len(candidate) <= max_characters:
            current.append(word)
        else:
            if current:
                lines.append(" ".join(current))
            current = [word]

    if current:
        lines.append(" ".join(current))

    return lines


def object_display_color(target_object):
    return {
        "battery": (0, 220, 255),
        "chemical_bottle": (40, 40, 255),
        "plastic_bottle": (40, 220, 40),
        "syringe": (255, 150, 40),
    }.get(target_object, (255, 255, 255))


def add_demo_side_panel(
    frame,
    instruction,
    status,
    target_object,
    target_bin,
    progress,
    success=None,
):
    """Place a compact information panel fully outside the simulation."""
    scene_h, scene_w = frame.shape[:2]
    panel_w = 256

    canvas = np.zeros(
        (scene_h, scene_w + panel_w, 3),
        dtype=np.uint8,
    )

    canvas[:, :scene_w] = frame
    panel_x = scene_w
    canvas[:, panel_x:] = (20, 23, 30)

    cv2.putText(
        canvas,
        "SafeSort-RL",
        (panel_x + 16, 34),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.70,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    cv2.putText(
        canvas,
        "Genesis | AMD ROCm",
        (panel_x + 16, 56),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.38,
        (175, 180, 190),
        1,
        cv2.LINE_AA,
    )

    cv2.putText(
        canvas,
        "INSTRUCTION",
        (panel_x + 16, 88),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )

    y = 110
    for line in wrap_text(instruction, 25):
        cv2.putText(
            canvas,
            line,
            (panel_x + 16, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.39,
            (245, 245, 245),
            1,
            cv2.LINE_AA,
        )
        y += 20

    y += 10
    cv2.putText(
        canvas,
        "TARGET",
        (panel_x + 16, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )
    y += 24

    cv2.putText(
        canvas,
        target_object.replace("_", " ").upper(),
        (panel_x + 16, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.49,
        object_display_color(target_object),
        2,
        cv2.LINE_AA,
    )

    y += 37
    cv2.putText(
        canvas,
        "DESTINATION",
        (panel_x + 16, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )
    y += 25

    destination_color = (
        (45, 225, 75)
        if target_bin == "recycle"
        else (45, 45, 245)
    )

    cv2.putText(
        canvas,
        f"{target_bin.upper()} BIN",
        (panel_x + 16, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.50,
        destination_color,
        2,
        cv2.LINE_AA,
    )

    y += 39
    cv2.putText(
        canvas,
        "PHASE",
        (panel_x + 16, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )
    y += 24

    for line in wrap_text(status, 25):
        cv2.putText(
            canvas,
            line,
            (panel_x + 16, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.39,
            (0, 245, 255),
            1,
            cv2.LINE_AA,
        )
        y += 19

    bar_x0 = panel_x + 16
    bar_x1 = panel_x + panel_w - 16
    bar_y = scene_h - 28

    cv2.rectangle(
        canvas,
        (bar_x0, bar_y),
        (bar_x1, bar_y + 10),
        (55, 60, 70),
        -1,
    )

    fill_x = int(
        bar_x0
        + np.clip(progress, 0.0, 1.0)
        * (bar_x1 - bar_x0)
    )

    cv2.rectangle(
        canvas,
        (bar_x0, bar_y),
        (fill_x, bar_y + 10),
        (70, 205, 255),
        -1,
    )

    if success is not None:
        label = "SUCCESS" if success else "FAILED"
        label_color = (
            (45, 230, 80)
            if success
            else (45, 45, 245)
        )

        cv2.putText(
            canvas,
            label,
            (panel_x + 16, scene_h - 44),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.62,
            label_color,
            2,
            cv2.LINE_AA,
        )

    return canvas

def make_title_card(width=976, height=400):
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    frame[:] = (19, 23, 31)
    cv2.putText(frame, "SafeSort-RL", (55, 108), cv2.FONT_HERSHEY_SIMPLEX, 1.55, (255,255,255), 3, cv2.LINE_AA)
    cv2.putText(frame, "Language-Conditioned Hazardous-Waste Sorting", (55, 165), cv2.FONT_HERSHEY_SIMPLEX, 0.68, (110,220,255), 2, cv2.LINE_AA)
    cv2.putText(frame, "Genesis World | Franka Panda | AMD Radeon ROCm", (55, 218), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (205,205,205), 1, cv2.LINE_AA)
    cv2.putText(frame, "81% success | 0% random baseline | 100 balanced episodes", (55, 276), cv2.FONT_HERSHEY_SIMPLEX, 0.56, (80,230,120), 2, cv2.LINE_AA)
    return frame


def make_results_card(width=976, height=400):
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    frame[:] = (19, 23, 31)
    cv2.putText(frame, "Evaluation on 100 balanced episodes", (48, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2, cv2.LINE_AA)
    rows = [
        ("Overall", "81%"),
        ("Battery -> hazard", "44%"),
        ("Chemical bottle -> hazard", "92%"),
        ("Plastic bottle -> recycle", "88%"),
        ("Syringe -> hazard", "100%"),
        ("Random policy", "0%"),
    ]
    y = 105
    for label, value in rows:
        cv2.putText(frame, label, (70, y), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (215,215,215), 1, cv2.LINE_AA)
        cv2.putText(frame, value, (760, y), cv2.FONT_HERSHEY_SIMPLEX, 0.64, (80,230,120), 2, cv2.LINE_AA)
        y += 46
    return frame


In [20]:
# ---------------------------------------------------------------------------
# SAFESORT-RL FINAL REALISTIC FOUR-OBJECT SHOWCASE WITH SIDE PANEL
# ---------------------------------------------------------------------------
# Uses the physically verified battery-v5 / remaining-objects-v2 motion logic.
#
# Visual improvements:
#   - synchronized phase/instruction side panel on every frame
#   - recognizable multi-part battery, chemical bottle, plastic bottle,
#     and syringe visual skins
#   - visual skins are shown only during camera rendering, then parked again
#     before the next physics step, so they do not disturb the verified grasp
#
# This cell does not retrain PPO and does not change evaluation results.
# It expects the existing physical-weld `base_env` from the final notebook.
# ---------------------------------------------------------------------------

from pathlib import Path
import time

import cv2
import imageio.v2 as imageio
import numpy as np

try:
    from IPython.display import Video, display
except Exception:
    Video = None
    display = None


# ---------------------------------------------------------------------------
# Verified object-specific physical settings.
# ---------------------------------------------------------------------------

SHOWCASE_CONFIG = {
    "battery": {
        "instruction": "Move the battery to the hazard bin.",
        "target_bin": "hazard",
        "start": np.array([0.51, -0.10, 0.1025], dtype=np.float32),
        "hand_minus_object": np.array([0.0, 0.0, 0.112], dtype=np.float32),
        "closed_opening": 0.017,
        "release_center_z": 0.150,
        "release_inward_y": 0.030,
        "minimum_final_z": 0.135,
        "display_name": "BATTERY",
    },
    "chemical_bottle": {
        "instruction": "Move the chemical bottle to the hazard bin.",
        "target_bin": "hazard",
        "start": np.array([0.51, -0.10, 0.1125], dtype=np.float32),
        "hand_minus_object": np.array([0.0, 0.0, 0.122], dtype=np.float32),
        "closed_opening": 0.022,
        "release_center_z": 0.160,
        "release_inward_y": 0.030,
        "minimum_final_z": 0.115,
        "display_name": "CHEMICAL BOTTLE",
    },
    "plastic_bottle": {
        "instruction": "Move the plastic bottle to the recycle bin.",
        "target_bin": "recycle",
        "start": np.array([0.51, -0.10, 0.1225], dtype=np.float32),
        "hand_minus_object": np.array([0.0, 0.0, 0.127], dtype=np.float32),
        "closed_opening": 0.021,
        "release_center_z": 0.170,
        "release_inward_y": 0.030,
        "minimum_final_z": 0.114,
        "display_name": "PLASTIC BOTTLE",
    },
    "syringe": {
        "instruction": "Move the syringe to the hazard bin.",
        "target_bin": "hazard",
        "start": np.array([0.51, -0.10, 0.1125], dtype=np.float32),
        "hand_minus_object": np.array([0.0, 0.0, 0.102], dtype=np.float32),
        "closed_opening": 0.011,
        "release_center_z": 0.160,
        "release_inward_y": 0.030,
        "minimum_final_z": 0.104,
        "display_name": "SYRINGE",
    },
}

SHOWCASE_OBJECT_ORDER = [
    "battery",
    "chemical_bottle",
    "plastic_bottle",
    "syringe",
]

SHOWCASE_OPENING = 0.040
SHOWCASE_FPS = 20
SHOWCASE_OUTPUT = Path(
    "videos/safesort_final_realistic_sidepanel_showcase.mp4"
)

# Verified empty-hand retreat values from remaining-objects v2.
SAFE_RETREAT_VERTICAL_CLEARANCE = 0.16
SAFE_RETREAT_MAX_HAND_Z = 0.46

# The fixed multi-part models already in env.objects were originally centered
# slightly higher than the smaller dynamic physical proxies. These offsets
# align their bottoms and make the rendered object sit naturally on the table.
REALISTIC_SKIN_CENTER_OFFSET = {
    "battery": np.array([0.0, 0.0, 0.0225], dtype=np.float32),
    "chemical_bottle": np.array([0.0, 0.0, 0.0175], dtype=np.float32),
    "plastic_bottle": np.array([0.0, 0.0, 0.0225], dtype=np.float32),
    "syringe": np.array([0.0, 0.0, 0.0325], dtype=np.float32),
}

# For the battery and syringe, the physical proxy itself is a useful main body,
# so only the identifying decorative parts are added. The two bottle skins use
# their complete multi-part models so their shoulders, necks, caps and labels
# are clearly visible.
REALISTIC_SKIN_PART_INDICES = {
    "battery": [1, 2, 3],
    "chemical_bottle": [0, 1, 2, 3, 4],
    "plastic_bottle": [0, 1, 2, 3, 4],
    "syringe": [1, 2, 3, 4],
}

REALISTIC_SKIN_HIDDEN_BASE = {
    object_type: np.array(
        [1.30 + 0.15 * index, -0.92, -0.35],
        dtype=np.float32,
    )
    for index, object_type in enumerate(SHOWCASE_OBJECT_ORDER)
}


# ---------------------------------------------------------------------------
# Conversion and quaternion helpers.
# ---------------------------------------------------------------------------

def _showcase_to_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()

    array = np.asarray(value)
    if array.ndim > 1:
        array = array[0]

    return array.astype(np.float32)


def _showcase_entity_position(entity):
    return _showcase_to_numpy(entity.get_pos())[:3]


def _showcase_entity_quaternion(entity):
    return _showcase_to_numpy(entity.get_quat())[:4]


def _showcase_hand_position(env):
    return _showcase_to_numpy(env.end_effector.get_pos())[:3]


def _showcase_active_object(env, object_type):
    return env.showcase_objects[object_type]


def _showcase_object_position(env, object_type):
    return _showcase_entity_position(
        _showcase_active_object(env, object_type)
    )


def _normalize_quaternion(quaternion):
    quaternion = np.asarray(quaternion, dtype=np.float32)
    norm = float(np.linalg.norm(quaternion))
    if norm < 1e-8:
        return np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)
    return quaternion / norm


def _rotate_vector_by_quaternion(vector, quaternion):
    """Rotate a 3-vector using a Genesis w-x-y-z quaternion."""
    w, x, y, z = _normalize_quaternion(quaternion)
    q_vector = np.array([x, y, z], dtype=np.float32)
    vector = np.asarray(vector, dtype=np.float32)

    uv = np.cross(q_vector, vector)
    uuv = np.cross(q_vector, uv)
    return vector + 2.0 * (w * uv + uuv)


def _safe_set_entity_pose(entity, position, quaternion):
    position = np.asarray(position, dtype=np.float32)
    quaternion = _normalize_quaternion(quaternion)

    try:
        entity.set_pos(position, zero_velocity=True)
    except TypeError:
        entity.set_pos(position)

    try:
        entity.set_quat(quaternion, zero_velocity=True)
    except TypeError:
        entity.set_quat(quaternion)


# ---------------------------------------------------------------------------
# Realistic render-skin helpers.
# The fixed detailed objects are never present during a physics step.
# ---------------------------------------------------------------------------

def _hide_realistic_skin(env, object_type):
    if not hasattr(env, "objects") or object_type not in env.objects:
        return

    group = env.objects[object_type]
    hidden_base = REALISTIC_SKIN_HIDDEN_BASE[object_type]
    identity = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)

    for part, offset in zip(group["parts"], group["offsets"]):
        _safe_set_entity_pose(
            part,
            hidden_base + np.asarray(offset, dtype=np.float32),
            identity,
        )


def _hide_all_realistic_skins(env):
    for object_type in SHOWCASE_OBJECT_ORDER:
        _hide_realistic_skin(env, object_type)


def _show_realistic_skin_for_render(env, object_type):
    """
    Place the existing detailed fixed model over the dynamic physical proxy.

    This is called immediately before camera.render(). The skin is parked again
    immediately afterward and no scene.step() occurs while it is visible.
    """
    if not hasattr(env, "objects") or object_type not in env.objects:
        return False

    proxy = _showcase_active_object(env, object_type)
    proxy_position = _showcase_entity_position(proxy)
    proxy_quaternion = _showcase_entity_quaternion(proxy)

    center_offset = _rotate_vector_by_quaternion(
        REALISTIC_SKIN_CENTER_OFFSET[object_type],
        proxy_quaternion,
    )
    skin_center = proxy_position + center_offset

    group = env.objects[object_type]
    selected_indices = set(REALISTIC_SKIN_PART_INDICES[object_type])

    for part_index, (part, local_offset) in enumerate(
        zip(group["parts"], group["offsets"])
    ):
        if part_index not in selected_indices:
            continue

        rotated_offset = _rotate_vector_by_quaternion(
            np.asarray(local_offset, dtype=np.float32),
            proxy_quaternion,
        )
        _safe_set_entity_pose(
            part,
            skin_center + rotated_offset,
            proxy_quaternion,
        )

    return True


# ---------------------------------------------------------------------------
# Side-panel drawing.
# ---------------------------------------------------------------------------

def _wrap_showcase_text(text, max_characters=25):
    words = str(text).split()
    lines = []
    current = []

    for word in words:
        candidate = " ".join(current + [word])
        if len(candidate) <= max_characters:
            current.append(word)
        else:
            if current:
                lines.append(" ".join(current))
            current = [word]

    if current:
        lines.append(" ".join(current))

    return lines


def _object_panel_color(object_type):
    return {
        "battery": (0, 220, 255),
        "chemical_bottle": (45, 55, 245),
        "plastic_bottle": (55, 220, 65),
        "syringe": (255, 165, 55),
    }.get(object_type, (245, 245, 245))


def _draw_object_icon(canvas, origin, object_type):
    x, y = origin
    color = _object_panel_color(object_type)

    if object_type == "battery":
        cv2.rectangle(canvas, (x + 8, y + 12), (x + 65, y + 62), color, 2)
        cv2.rectangle(canvas, (x + 19, y + 5), (x + 29, y + 12), color, -1)
        cv2.rectangle(canvas, (x + 47, y + 5), (x + 56, y + 12), color, -1)
        cv2.putText(canvas, "+", (x + 17, y + 46), cv2.FONT_HERSHEY_SIMPLEX, 0.50, color, 2, cv2.LINE_AA)
        cv2.putText(canvas, "-", (x + 47, y + 45), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    elif object_type in {"chemical_bottle", "plastic_bottle"}:
        cv2.rectangle(canvas, (x + 19, y + 25), (x + 58, y + 65), color, 2)
        cv2.rectangle(canvas, (x + 25, y + 17), (x + 52, y + 28), color, 2)
        cv2.rectangle(canvas, (x + 31, y + 8), (x + 46, y + 19), color, 2)
        cv2.rectangle(canvas, (x + 27, y + 39), (x + 50, y + 52), (235, 235, 235), 1)
        if object_type == "chemical_bottle":
            cv2.line(canvas, (x + 32, y + 43), (x + 45, y + 49), (20, 20, 20), 1)
            cv2.line(canvas, (x + 45, y + 43), (x + 32, y + 49), (20, 20, 20), 1)

    elif object_type == "syringe":
        cv2.rectangle(canvas, (x + 14, y + 25), (x + 52, y + 45), color, 2)
        cv2.line(canvas, (x + 8, y + 18), (x + 8, y + 52), color, 2)
        cv2.line(canvas, (x + 8, y + 35), (x + 14, y + 35), color, 2)
        cv2.line(canvas, (x + 52, y + 35), (x + 75, y + 35), color, 2)
        cv2.line(canvas, (x + 75, y + 35), (x + 84, y + 31), (210, 210, 215), 1)
        for mark_x in range(x + 20, x + 49, 7):
            cv2.line(canvas, (mark_x, y + 27), (mark_x, y + 32), color, 1)


def _add_showcase_side_panel(frame, context):
    scene_height, scene_width = frame.shape[:2]
    panel_width = 256

    canvas = np.zeros(
        (scene_height, scene_width + panel_width, 3),
        dtype=np.uint8,
    )
    canvas[:, :scene_width] = frame

    panel_x = scene_width
    canvas[:, panel_x:] = (20, 23, 30)

    cv2.putText(
        canvas,
        "SafeSort-RL",
        (panel_x + 16, 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.68,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        canvas,
        "Genesis | AMD ROCm",
        (panel_x + 16, 53),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.37,
        (175, 180, 190),
        1,
        cv2.LINE_AA,
    )

    task_index = int(context.get("task_index", 1))
    task_count = int(context.get("task_count", 4))
    cv2.putText(
        canvas,
        f"TASK {task_index} / {task_count}",
        (panel_x + 164, 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.36,
        (160, 220, 255),
        1,
        cv2.LINE_AA,
    )

    cv2.putText(
        canvas,
        "INSTRUCTION",
        (panel_x + 16, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.39,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )

    text_y = 101
    for line in _wrap_showcase_text(context["instruction"], 25):
        cv2.putText(
            canvas,
            line,
            (panel_x + 16, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            (245, 245, 245),
            1,
            cv2.LINE_AA,
        )
        text_y += 19

    icon_y = text_y + 3
    _draw_object_icon(
        canvas,
        (panel_x + 16, icon_y),
        context["object_type"],
    )

    cv2.putText(
        canvas,
        context["display_name"],
        (panel_x + 108, icon_y + 27),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.40,
        _object_panel_color(context["object_type"]),
        1,
        cv2.LINE_AA,
    )

    destination_color = (
        (45, 225, 75)
        if context["target_bin"] == "recycle"
        else (45, 45, 245)
    )
    cv2.putText(
        canvas,
        f"TO {context['target_bin'].upper()} BIN",
        (panel_x + 108, icon_y + 51),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.39,
        destination_color,
        1,
        cv2.LINE_AA,
    )

    phase_heading_y = icon_y + 87
    cv2.putText(
        canvas,
        "CURRENT PHASE",
        (panel_x + 16, phase_heading_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.39,
        (80, 210, 255),
        1,
        cv2.LINE_AA,
    )

    phase_y = phase_heading_y + 22
    for line in _wrap_showcase_text(context.get("phase", ""), 25):
        cv2.putText(
            canvas,
            line,
            (panel_x + 16, phase_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.40,
            (0, 245, 255),
            1,
            cv2.LINE_AA,
        )
        phase_y += 19

    attachment = context.get("attachment", "OPEN")
    cv2.putText(
        canvas,
        f"GRIP: {attachment}",
        (panel_x + 16, min(phase_y + 15, scene_height - 78)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.36,
        (190, 195, 205),
        1,
        cv2.LINE_AA,
    )

    if context.get("success") is not None:
        success = bool(context["success"])
        label = "SUCCESS" if success else "FAILED"
        label_color = (45, 230, 80) if success else (45, 45, 245)
        cv2.putText(
            canvas,
            label,
            (panel_x + 16, scene_height - 48),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.62,
            label_color,
            2,
            cv2.LINE_AA,
        )

    progress = float(np.clip(context.get("progress", 0.0), 0.0, 1.0))
    bar_x0 = panel_x + 16
    bar_x1 = panel_x + panel_width - 16
    bar_y = scene_height - 28

    cv2.rectangle(
        canvas,
        (bar_x0, bar_y),
        (bar_x1, bar_y + 10),
        (55, 60, 70),
        -1,
    )
    fill_x = int(bar_x0 + progress * (bar_x1 - bar_x0))
    cv2.rectangle(
        canvas,
        (bar_x0, bar_y),
        (fill_x, bar_y + 10),
        (70, 205, 255),
        -1,
    )

    cv2.putText(
        canvas,
        f"{int(round(progress * 100)):d}%",
        (bar_x1 - 37, bar_y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.32,
        (180, 185, 195),
        1,
        cv2.LINE_AA,
    )

    return canvas


def _make_showcase_title_card(width=960, height=432):
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    frame[:] = (19, 23, 31)

    cv2.putText(frame, "SafeSort-RL", (48, 104), cv2.FONT_HERSHEY_SIMPLEX, 1.55, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(frame, "Physical Four-Object Sorting Showcase", (48, 162), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (105, 220, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, "Franka Panda | Genesis | AMD ROCm", (48, 211), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (205, 205, 210), 1, cv2.LINE_AA)
    cv2.putText(frame, "Verified grasp, transport, safe release and retreat", (48, 265), cv2.FONT_HERSHEY_SIMPLEX, 0.54, (80, 230, 120), 2, cv2.LINE_AA)
    cv2.putText(frame, "Realistic object skins + synchronized phase panel", (48, 306), cv2.FONT_HERSHEY_SIMPLEX, 0.51, (215, 220, 230), 1, cv2.LINE_AA)
    cv2.putText(frame, "Evaluation remains based on the trained PPO checkpoint", (48, 362), cv2.FONT_HERSHEY_SIMPLEX, 0.47, (170, 175, 185), 1, cv2.LINE_AA)

    return frame


def _make_showcase_results_card(width=960, height=432):
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    frame[:] = (19, 23, 31)

    cv2.putText(frame, "SafeSort-RL Showcase Complete", (48, 68), cv2.FONT_HERSHEY_SIMPLEX, 0.90, (255, 255, 255), 2, cv2.LINE_AA)

    rows = [
        ("Battery", "HAZARD", "PASS"),
        ("Chemical bottle", "HAZARD", "PASS"),
        ("Plastic bottle", "RECYCLE", "PASS"),
        ("Syringe", "HAZARD", "PASS"),
    ]

    y = 132
    for name, destination, result in rows:
        cv2.putText(frame, name, (70, y), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (220, 220, 225), 1, cv2.LINE_AA)
        cv2.putText(frame, destination, (470, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (100, 205, 255), 1, cv2.LINE_AA)
        cv2.putText(frame, result, (770, y), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (70, 230, 110), 2, cv2.LINE_AA)
        y += 55

    cv2.putText(frame, "All objects remained inside their correct bins", (70, 372), cv2.FONT_HERSHEY_SIMPLEX, 0.56, (70, 230, 110), 2, cv2.LINE_AA)
    return frame


# ---------------------------------------------------------------------------
# Persistent Genesis controller helpers proven by the successful diagnostics.
# ---------------------------------------------------------------------------

def _showcase_delete_weld_without_step(env):
    if hasattr(env, "_delete_showcase_weld"):
        env._delete_showcase_weld()
    else:
        env.release_showcase_weld()


def _showcase_apply_joint_hold(env):
    hold_qpos = getattr(env, "_final_showcase_joint_hold_qpos", None)
    if hold_qpos is None:
        return

    env.franka.control_dofs_position(
        position=hold_qpos[:7],
        dofs_idx_local=env.arm_dofs,
    )
    env.franka.control_dofs_position(
        position=hold_qpos[-2:],
        dofs_idx_local=env.finger_dofs,
    )


def _showcase_step_scene(env, steps):
    for _ in range(int(steps)):
        _showcase_apply_joint_hold(env)
        env.scene.step()


def _showcase_set_robot_pose_exact(
    env,
    hand_position,
    finger_opening,
    settle_steps=6,
):
    hand_position = np.asarray(hand_position, dtype=np.float32)

    qpos = np.asarray(
        env._solve_ik(hand_position),
        dtype=np.float32,
    ).copy()
    qpos[-2:] = float(finger_opening)

    env.gripper_position = hand_position.copy()
    env.gripper_opening = float(finger_opening)

    env.franka.set_qpos(qpos, zero_velocity=True)

    env._final_showcase_joint_hold_qpos = qpos.copy()
    _showcase_apply_joint_hold(env)
    _showcase_step_scene(env, settle_steps)

    rendered_hand = _showcase_hand_position(env)
    pose_error = float(np.linalg.norm(rendered_hand - hand_position))

    if pose_error > 0.020:
        raise RuntimeError(
            "Held IK pose did not place the rendered hand at the "
            "requested point.\n"
            f"Requested: {hand_position}\n"
            f"Rendered: {rendered_hand}\n"
            f"Error: {pose_error:.4f} m"
        )


def _showcase_render(
    env,
    writer,
    context,
    camera="wide",
    repeats=1,
):
    object_type = context["object_type"]

    skin_visible = _show_realistic_skin_for_render(env, object_type)
    try:
        raw_frame = env.render(view=camera)
    finally:
        if skin_visible:
            _hide_realistic_skin(env, object_type)

    frame = _add_showcase_side_panel(raw_frame, context)

    for _ in range(max(1, int(repeats))):
        writer.append_data(frame)


def _showcase_hold(
    env,
    writer,
    context,
    seconds,
    camera="wide",
):
    _showcase_render(
        env,
        writer,
        context,
        camera=camera,
        repeats=max(1, int(seconds * SHOWCASE_FPS)),
    )


def _showcase_move_exact(
    env,
    writer,
    context,
    start_position,
    end_position,
    finger_opening,
    steps,
    phase,
    progress_start,
    progress_end,
    settle_steps=4,
    held_entity=None,
    held_position=None,
    camera="wide",
):
    start_position = np.asarray(start_position, dtype=np.float32)
    end_position = np.asarray(end_position, dtype=np.float32)

    for alpha in np.linspace(
        0.0,
        1.0,
        int(steps) + 1,
        dtype=np.float32,
    )[1:]:
        position = (
            start_position
            + alpha * (end_position - start_position)
        ).astype(np.float32)

        _showcase_set_robot_pose_exact(
            env,
            position,
            finger_opening,
            settle_steps=settle_steps,
        )

        if held_entity is not None and held_position is not None:
            held_entity.set_pos(
                np.asarray(held_position, dtype=np.float32),
                zero_velocity=True,
            )

        context["phase"] = phase
        context["progress"] = float(
            progress_start
            + alpha * (progress_end - progress_start)
        )

        _showcase_render(
            env,
            writer,
            context,
            camera=camera,
        )


def _showcase_close_gripper(
    env,
    writer,
    context,
    hand_position,
    closed_opening,
):
    for step_index, opening in enumerate(
        np.linspace(
            SHOWCASE_OPENING,
            float(closed_opening),
            24,
            dtype=np.float32,
        )
    ):
        _showcase_set_robot_pose_exact(
            env,
            hand_position,
            float(opening),
            settle_steps=3,
        )

        alpha = (step_index + 1) / 24.0
        context["phase"] = "GRASP - fingers closing around object"
        context["progress"] = 0.30 + 0.10 * alpha
        context["attachment"] = "CLOSING"

        _showcase_render(
            env,
            writer,
            context,
            camera="grasp",
        )


def _showcase_open_gripper_with_stabilized_object(
    env,
    writer,
    context,
    hand_position,
    active_object,
    release_position,
    closed_opening,
):
    for step_index, opening in enumerate(
        np.linspace(
            float(closed_opening),
            SHOWCASE_OPENING,
            28,
            dtype=np.float32,
        )
    ):
        _showcase_set_robot_pose_exact(
            env,
            hand_position,
            float(opening),
            settle_steps=2,
        )
        active_object.set_pos(
            release_position,
            zero_velocity=True,
        )

        alpha = (step_index + 1) / 28.0
        context["phase"] = "RELEASE - fingers opening inside bin"
        context["progress"] = 0.84 + 0.07 * alpha
        context["attachment"] = "RELEASED"

        _showcase_render(
            env,
            writer,
            context,
            camera="bin",
        )


def _showcase_hide_evaluation_objects(env):
    # Keep every detailed fixed model outside the cameras during physics.
    _hide_all_realistic_skins(env)


def _showcase_prepare_scene(env, object_type, config, seed):
    env.reset(
        seed=seed,
        options={"instruction": config["instruction"]},
    )

    env._final_showcase_joint_hold_qpos = None
    env.release_showcase_weld()
    env.park_showcase_objects()
    _showcase_hide_evaluation_objects(env)

    active_object = _showcase_active_object(env, object_type)
    active_object.set_pos(config["start"], zero_velocity=True)
    active_object.set_quat(
        np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32),
        zero_velocity=True,
    )

    # Let only the dynamic physical proxy settle. Detailed skins remain parked.
    _showcase_step_scene(env, 160)

    home = np.array([0.56, -0.28, 0.50], dtype=np.float32)
    _showcase_set_robot_pose_exact(
        env,
        home,
        SHOWCASE_OPENING,
        settle_steps=10,
    )

    return home


def _showcase_is_inside_bin(final_position, bin_center, minimum_final_z):
    return bool(
        abs(final_position[0] - bin_center[0]) <= 0.080
        and (
            bin_center[1] - 0.055
            <= final_position[1]
            <= bin_center[1] + 0.065
        )
        and final_position[2] >= float(minimum_final_z)
    )


# ---------------------------------------------------------------------------
# Record one complete object sequence into an already-open video writer.
# ---------------------------------------------------------------------------

def _record_showcase_object(
    env,
    writer,
    object_type,
    task_index,
    seed,
):
    config = SHOWCASE_CONFIG[object_type]
    home = _showcase_prepare_scene(env, object_type, config, seed)

    context = {
        "instruction": config["instruction"],
        "object_type": object_type,
        "display_name": config["display_name"],
        "target_bin": config["target_bin"],
        "task_index": task_index,
        "task_count": len(SHOWCASE_OBJECT_ORDER),
        "phase": "TASK READY - object identified",
        "progress": 0.02,
        "attachment": "OPEN",
        "success": None,
    }

    object_position = _showcase_object_position(env, object_type)
    grasp_hand_position = (
        object_position + config["hand_minus_object"]
    ).astype(np.float32)
    approach_hand_position = (
        grasp_hand_position
        + np.array([0.0, 0.0, 0.16], dtype=np.float32)
    )

    print("\n" + "=" * 76)
    print(f"Final showcase task {task_index}/4: {object_type}")
    print("Instruction:", config["instruction"])
    print("Object start:", np.round(object_position, 4))

    _showcase_hold(
        env,
        writer,
        context,
        seconds=1.2,
        camera="wide",
    )

    _showcase_move_exact(
        env,
        writer,
        context,
        home,
        approach_hand_position,
        SHOWCASE_OPENING,
        steps=28,
        phase="APPROACH - moving above selected object",
        progress_start=0.04,
        progress_end=0.18,
        camera="wide",
    )

    context["phase"] = "APPROACH COMPLETE - directly above object"
    context["progress"] = 0.18
    _showcase_hold(env, writer, context, 0.5, camera="wide")

    _showcase_move_exact(
        env,
        writer,
        context,
        approach_hand_position,
        grasp_hand_position,
        SHOWCASE_OPENING,
        steps=24,
        phase="ALIGN - lowering open fingers around object",
        progress_start=0.18,
        progress_end=0.30,
        camera="grasp",
    )

    alignment_error = float(
        np.linalg.norm(
            _showcase_hand_position(env) - grasp_hand_position
        )
    )
    print("Rendered hand error:", f"{alignment_error:.4f} m")

    context["phase"] = "ALIGNMENT COMPLETE - object between fingers"
    context["progress"] = 0.30
    _showcase_hold(env, writer, context, 0.8, camera="grasp")

    _showcase_close_gripper(
        env,
        writer,
        context,
        grasp_hand_position,
        config["closed_opening"],
    )

    env.add_showcase_weld(object_type)
    _showcase_step_scene(env, 80)

    context["phase"] = "GRASP COMPLETE - physical attachment engaged"
    context["progress"] = 0.42
    context["attachment"] = "WELDED"
    _showcase_hold(env, writer, context, 0.7, camera="grasp")

    # Verified 10 cm lift test using actual hand and object motion.
    test_lift_position = (
        grasp_hand_position
        + np.array([0.0, 0.0, 0.10], dtype=np.float32)
    )

    hand_before = _showcase_hand_position(env)
    object_before = _showcase_object_position(env, object_type)
    relative_before = hand_before - object_before

    _showcase_move_exact(
        env,
        writer,
        context,
        grasp_hand_position,
        test_lift_position,
        config["closed_opening"],
        steps=30,
        settle_steps=10,
        phase="LIFT TEST - verifying object follows hand",
        progress_start=0.42,
        progress_end=0.52,
        camera="grasp",
    )

    _showcase_step_scene(env, 180)
    context["phase"] = "LIFT VERIFIED - object securely attached"
    context["progress"] = 0.52
    _showcase_hold(env, writer, context, 0.6, camera="grasp")

    hand_after = _showcase_hand_position(env)
    object_after = _showcase_object_position(env, object_type)
    relative_after = hand_after - object_after

    hand_displacement = hand_after - hand_before
    object_displacement = object_after - object_before
    hand_lift = float(hand_displacement[2])
    object_lift = float(object_displacement[2])
    offset_drift = float(np.linalg.norm(relative_after - relative_before))
    displacement_error = float(
        np.linalg.norm(object_displacement - hand_displacement)
    )

    minimum_object_lift = max(0.015, 0.45 * hand_lift)
    if (
        hand_lift < 0.040
        or object_lift < minimum_object_lift
        or offset_drift > 0.040
        or displacement_error > 0.045
    ):
        raise RuntimeError(
            f"{object_type} failed the verified weld-follow test.\n"
            f"Hand lift: {hand_lift:.4f} m\n"
            f"Object lift: {object_lift:.4f} m\n"
            f"Offset drift: {offset_drift:.4f} m\n"
            f"Displacement error: {displacement_error:.4f} m"
        )

    print(
        "Weld-follow:",
        f"hand={hand_lift:.4f} m,",
        f"object={object_lift:.4f} m,",
        f"drift={offset_drift:.4f} m",
    )

    lifted_hand_position = (
        grasp_hand_position
        + np.array([0.0, 0.0, 0.24], dtype=np.float32)
    )
    _showcase_move_exact(
        env,
        writer,
        context,
        test_lift_position,
        lifted_hand_position,
        config["closed_opening"],
        steps=26,
        settle_steps=6,
        phase="LIFT - raising object to transport height",
        progress_start=0.52,
        progress_end=0.60,
        camera="wide",
    )

    target_bin_name = config["target_bin"]
    bin_center = np.asarray(
        env.bin_positions[target_bin_name],
        dtype=np.float32,
    )

    held_offset = (
        _showcase_hand_position(env)
        - _showcase_object_position(env, object_type)
    ).astype(np.float32)

    desired_object_above_bin = np.array(
        [bin_center[0], bin_center[1], 0.34],
        dtype=np.float32,
    )
    hand_above_bin = (
        desired_object_above_bin + held_offset
    ).astype(np.float32)

    _showcase_move_exact(
        env,
        writer,
        context,
        lifted_hand_position,
        hand_above_bin,
        config["closed_opening"],
        steps=36,
        settle_steps=6,
        phase=f"TRANSPORT - carrying object to {target_bin_name} bin",
        progress_start=0.60,
        progress_end=0.72,
        camera="wide",
    )

    context["phase"] = f"POSITIONED - above {target_bin_name} bin"
    context["progress"] = 0.72
    _showcase_hold(env, writer, context, 0.6, camera="bin")

    safe_release_xy = (
        bin_center[:2]
        + np.array(
            [0.0, config["release_inward_y"]],
            dtype=np.float32,
        )
    )
    desired_object_release = np.array(
        [
            safe_release_xy[0],
            safe_release_xy[1],
            config["release_center_z"],
        ],
        dtype=np.float32,
    )
    hand_release_position = (
        desired_object_release + held_offset
    ).astype(np.float32)

    _showcase_move_exact(
        env,
        writer,
        context,
        hand_above_bin,
        hand_release_position,
        config["closed_opening"],
        steps=30,
        settle_steps=6,
        phase="LOWER - placing object deep inside bin",
        progress_start=0.72,
        progress_end=0.84,
        camera="bin",
    )

    object_before_release = _showcase_object_position(env, object_type)
    release_error = float(
        np.linalg.norm(object_before_release[:2] - safe_release_xy)
    )
    print("Safe release error:", f"{release_error:.4f} m")

    if release_error > 0.040:
        raise RuntimeError(
            f"{object_type} did not reach the safe interior release point.\n"
            f"Safe-target error: {release_error:.4f} m"
        )

    context["phase"] = "PLACEMENT READY - object supported inside bin"
    context["progress"] = 0.84
    _showcase_hold(env, writer, context, 0.7, camera="bin")

    active_object = _showcase_active_object(env, object_type)
    _showcase_delete_weld_without_step(env)

    active_object.set_pos(
        desired_object_release,
        zero_velocity=True,
    )
    active_object.set_quat(
        np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32),
        zero_velocity=True,
    )

    _showcase_open_gripper_with_stabilized_object(
        env,
        writer,
        context,
        hand_release_position,
        active_object,
        desired_object_release,
        config["closed_opening"],
    )

    retreat_up = hand_release_position.copy()
    retreat_up[2] = min(
        float(hand_release_position[2])
        + SAFE_RETREAT_VERTICAL_CLEARANCE,
        SAFE_RETREAT_MAX_HAND_Z,
    )

    _showcase_move_exact(
        env,
        writer,
        context,
        hand_release_position,
        retreat_up,
        SHOWCASE_OPENING,
        steps=24,
        held_entity=active_object,
        held_position=desired_object_release,
        phase="RETREAT - lifting open gripper away from bin",
        progress_start=0.91,
        progress_end=0.96,
        camera="bin",
    )

    active_object.set_pos(
        desired_object_release,
        zero_velocity=True,
    )

    context["phase"] = "SETTLING - object resting under bin physics"
    context["progress"] = 0.97
    context["attachment"] = "OPEN"

    for settle_index in range(int(1.2 * SHOWCASE_FPS)):
        _showcase_step_scene(env, 4)
        context["progress"] = 0.97 + 0.02 * (
            (settle_index + 1) / (1.2 * SHOWCASE_FPS)
        )
        _showcase_render(env, writer, context, camera="bin")

    # Return the empty hand home before declaring final success. The object is
    # left under normal bin physics during this motion, matching the successful
    # diagnostics.
    retreat_home = np.array([0.50, -0.36, 0.52], dtype=np.float32)
    context["success"] = None
    _showcase_move_exact(
        env,
        writer,
        context,
        retreat_up,
        retreat_home,
        SHOWCASE_OPENING,
        steps=30,
        phase="RETREAT - arm returning home",
        progress_start=0.99,
        progress_end=1.0,
        camera="wide",
    )

    final_position = _showcase_object_position(env, object_type)
    inside = _showcase_is_inside_bin(
        final_position,
        bin_center,
        config["minimum_final_z"],
    )

    if not inside:
        context["phase"] = "FAILED - object left destination bin"
        context["success"] = False
        _showcase_hold(env, writer, context, 0.8, camera="bin")
        raise RuntimeError(
            f"{object_type} left the safe interior of the "
            f"{target_bin_name} bin after release.\n"
            f"Final position: {final_position}"
        )

    context["phase"] = f"SUCCESS - {config['display_name'].lower()} secured in {target_bin_name} bin"
    context["progress"] = 1.0
    context["success"] = True
    context["attachment"] = "OPEN"
    _showcase_hold(env, writer, context, 1.5, camera="bin")

    print("Final position:", np.round(final_position, 4))
    print(f"Remained inside {target_bin_name} bin:", inside)

    return {
        "object_type": object_type,
        "target_bin": target_bin_name,
        "final_position": final_position.copy(),
        "inside": inside,
    }


# ---------------------------------------------------------------------------
# Record the complete four-object final video.
# ---------------------------------------------------------------------------

def record_final_realistic_sidepanel_showcase(
    env=base_env,
    output_path=SHOWCASE_OUTPUT,
    fps=SHOWCASE_FPS,
    display_video=True,
):
    required_attributes = [
        "showcase_objects",
        "objects",
        "park_showcase_objects",
        "add_showcase_weld",
        "release_showcase_weld",
        "_solve_ik",
        "end_effector",
        "arm_dofs",
        "finger_dofs",
        "bin_positions",
    ]
    missing = [
        name for name in required_attributes
        if not hasattr(env, name)
    ]
    if missing:
        raise RuntimeError(
            "base_env is not the required final physical-weld environment.\n"
            f"Missing: {missing}"
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    started = time.perf_counter()
    results = []

    with imageio.get_writer(
        output_path,
        fps=int(fps),
        macro_block_size=16,
    ) as writer:
        title_card = _make_showcase_title_card()
        for _ in range(int(2.8 * fps)):
            writer.append_data(title_card)

        for task_index, object_type in enumerate(
            SHOWCASE_OBJECT_ORDER,
            start=1,
        ):
            result = _record_showcase_object(
                env=env,
                writer=writer,
                object_type=object_type,
                task_index=task_index,
                seed=140_000 + task_index,
            )
            results.append(result)

        results_card = _make_showcase_results_card()
        for _ in range(int(3.0 * fps)):
            writer.append_data(results_card)

    elapsed = time.perf_counter() - started

    print("\nFinal realistic side-panel showcase complete")
    print("Saved:", output_path)
    print("Generation time:", f"{elapsed:.1f} seconds")
    for result in results:
        print(
            f"  {result['object_type']} -> {result['target_bin']}: ",
            result["inside"],
        )

    if display_video and display is not None and Video is not None:
        display(Video(str(output_path), embed=True))

    return str(output_path), results


final_realistic_showcase_path, final_realistic_showcase_results = (
    record_final_realistic_sidepanel_showcase(
        env=base_env,
    )
)


Final showcase task 1/4: battery
Instruction: Move the battery to the hazard bin.
Object start: [ 0.51   -0.1     0.1025]
Rendered hand error: 0.0002 m
Weld-follow: hand=0.0870 m, object=0.0869 m, drift=0.0041 m
Safe release error: 0.0123 m
Final position: [0.72   0.2602 0.1425]
Remained inside hazard bin: True

Final showcase task 2/4: chemical_bottle
Instruction: Move the chemical bottle to the hazard bin.
Object start: [ 0.51   -0.1     0.1125]
Rendered hand error: 0.0002 m
Weld-follow: hand=0.0871 m, object=0.0870 m, drift=0.0043 m
Safe release error: 0.0009 m
Final position: [0.72   0.2601 0.1525]
Remained inside hazard bin: True

Final showcase task 3/4: plastic_bottle
Instruction: Move the plastic bottle to the recycle bin.
Object start: [ 0.51   -0.1     0.1225]
Rendered hand error: 0.0001 m
Weld-follow: hand=0.0865 m, object=0.0864 m, drift=0.0047 m
Safe release error: 0.0007 m
Final position: [0.43   0.2601 0.1625]
Remained inside recycle bin: True

Final showcase task 4/4: 

In [21]:
expected_video_path = Path(
    "videos/safesort_final_realistic_sidepanel_showcase.mp4"
)

if expected_video_path.exists():
    print("✓ Physical-weld showcase generated")
    print(expected_video_path.resolve())
    print(
        f"Size: "
        f"{expected_video_path.stat().st_size / (1024 * 1024):.1f} MB"
    )
else:
    print("⚠ Physical-weld showcase not found")
    print(
        "Confirm RUN_VIDEO=True, the final checkpoint loaded, "
        "and the showcase cell completed."
    )


✓ Physical-weld showcase generated
/workspace/videos/safesort_final_realistic_sidepanel_showcase.mp4
Size: 1.8 MB


## 18. Export the final results

The main project information is collected into one JSON file:

```text
results/final_evaluation.json
```

The export includes the selected checkpoint, AMD/ROCm environment, overall and per-task success rates, random-policy baseline, instruction-swap output, and any fresh evaluation results produced during the current run.

This file is useful for reports, grading, and later comparison because the important numbers do not have to be copied manually from notebook output.


In [22]:
Path("results").mkdir(exist_ok=True)

submission_results = {
    "project": "SafeSort-RL",
    "simulator": "Genesis World 1.3.1",
    "robot": "Franka Panda",
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Unavailable"
    ),
    "hip_version": torch.version.hip,
    "policy": "PPO MlpPolicy",
    "selected_checkpoint": (
        selected_model_path + ".zip"
        if selected_model_path is not None
        else None
    ),
    "overall_success_rate": 0.81,
    "mean_reward": 38.930,
    "reward_std": 19.871,
    "mean_episode_length": 70.1,
    "mean_final_bin_distance": 0.158,
    "per_task_success": {
        "battery_to_hazard": 0.44,
        "chemical_bottle_to_hazard": 0.92,
        "plastic_bottle_to_recycle": 0.88,
        "syringe_to_hazard": 1.00,
    },
    "random_policy_success_rate": 0.0,
    "random_policy_mean_reward": -1.509,
    "amd_environment": amd_environment,
    "instruction_swap_probe": instruction_swap_results,
    "fresh_evaluation": final_results,
}

results_path = Path(
    "results/final_evaluation.json"
)

results_path.write_text(
    json.dumps(
        submission_results,
        indent=2,
    ),
    encoding="utf-8",
)

print("✓ Saved:", results_path)


✓ Saved: results/final_evaluation.json


## 19. Project summary

### The problem

Batteries, chemical containers, and used syringes can expose workers to toxic, corrosive, or biohazardous material. SafeSort-RL studies whether a simulated robot can use a task instruction to move these objects into hazard or recycling bins.

### The approach

A PPO policy receives compact robot and object state together with a one-hot task instruction. It predicts small Cartesian hand movements. Genesis inverse kinematics converts those movements into Franka Panda joint targets, while a deterministic safety layer handles attachment and release.

### AMD / ROCm work

The complete Genesis scene runs on an AMD Radeon GPU through the `gs.amdgpu` backend. The notebook records the active HIP version, GPU identity, Genesis device, and software versions, with optional `rocm-smi` and throughput measurements.

### Final result

The balanced model achieved **81% overall success across 100 episodes**, compared with **0% for the random-action baseline**.

### What still needs improvement

- Battery sorting remains the weakest task at 44%.
- Instructions are symbolic and rule-normalized rather than grounded by a pretrained language model.
- The policy uses state vectors instead of learning directly from RGB images.
- Grasp attachment and release are deterministic rather than learned end to end.
- Results are limited to simulation; no real Franka experiment has been completed.

### Logical next steps

A stronger follow-up would add camera-based perception, domain randomization, learned grasp confidence, and real-robot validation. The battery task is the clearest place to begin because it remains the largest source of failure in the balanced evaluation.


## 20. Final submission checklist

Use this list before packaging the project.

### Runtime

- [ ] The **Genesis AMD ROCm** kernel is selected.
- [ ] PyTorch reports a HIP version and the AMD GPU.
- [ ] Genesis initializes with `gs.amdgpu`.
- [ ] The scene builds successfully.
- [ ] The smoke test passes.

### Required project files

- [ ] The final notebook is included.
- [ ] `models/safesort_genesis_amd_final_best.zip` is present.
- [ ] `README.md` and `requirements.txt` are included.
- [ ] The final results JSON is present.
- [ ] The final showcase MP4 is present.

### Evidence and reporting

- [ ] The 81% balanced evaluation result is reported accurately.
- [ ] The 0% random-policy baseline is included.
- [ ] AMD/ROCm environment evidence is saved.
- [ ] The deterministic grasp/release limitation is stated clearly.
- [ ] The showcase is described as a deterministic presentation, separate from PPO evaluation.

### Expected final outputs

```text
results/final_evaluation.json
results/amd_environment.json
results/amd_environment.txt
videos/safesort_final_realistic_sidepanel_showcase.mp4
```


In [23]:
print("\nFINAL STATUS")
print("  Genesis backend:", gs.backend)
print("  Genesis device:", gs.device)
print("  GPU:", torch.cuda.get_device_name(0))
print("  HIP:", torch.version.hip)
print(
    "  Checkpoint loaded:",
    final_model is not None,
)
print(
    "  Video generated:",
    expected_video_path.exists(),
)
print("  Verified overall success: 81%")
print("\n✓ Final submission notebook completed")



FINAL STATUS
  Genesis backend: 3
  Genesis device: cuda:0
  GPU: AMD Radeon Graphics
  HIP: 6.4.43482-0f2d60242
  Checkpoint loaded: True
  Video generated: True
  Verified overall success: 81%

✓ Final submission notebook completed
